In [ ]:
"""
Unified Trading Bot
═══════════════════
Telegram Poller  ──►  MongoDB (watchlist_tickers)  ──►  IBKR Trading Loop

Flow:
  1. Telethon polls a private Telegram channel every 20 seconds.
  2. NEW signals are parsed for a ticker symbol and upserted into
     the 'watchlist_tickers' MongoDB collection.
  3. The IBKR trading loop runs every 5 minutes, reads ALL active
     tickers from 'watchlist_tickers', qualifies contracts on-the-fly,
     and evaluates buy / position-management logic for each.
  4. At EOD (>= 4 PM) the watchlist_tickers collection is cleared.

ENTRY SCORING (v2):
  Each ticker is scored on 3 conditions (1 point each):
    1. Parabolic SAR indicates bullish entry (SAR below price)
    2. Volume Heatmap shows "Extra High Volume Up"
    3. Close > EMA 9 > EMA 21  (bullish EMA stack)
  Score >= 2  →  enter position

EVENT LOOP:
  Both Telethon and ib_insync share the SAME asyncio event loop via
  nest_asyncio. The Telegram poller and the trading loop run as
  concurrent asyncio tasks — no threads needed.
"""

import asyncio
import os
import re
import sys
import random
import logging
import warnings
from datetime import datetime, timezone, timedelta

import nest_asyncio
nest_asyncio.apply()          # must be before any IB / asyncio.run() usage

import numpy as np
import pandas as pd
from pymongo import MongoClient
from bson import ObjectId

from ib_insync import IB, Stock, MarketOrder, util

from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import FloodWaitError, UserAlreadyParticipantError
from telethon.tl.functions.messages import ImportChatInviteRequest

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt    = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh     = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh     = logging.FileHandler("unified_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("unified_bot")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


def safe_print(text: str):
    try:
        encoded = text.encode(sys.stdout.encoding or "utf-8", errors="replace")
        sys.stdout.buffer.write(encoded + b"\n")
        sys.stdout.buffer.flush()
    except Exception:
        print(text)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

# Telegram
API_ID        = 34025130
API_HASH      = "e5e514ebcb8a15042feadfecd7a15cfe"
PHONE         = "+18477675748"
INVITE_LINK   = "https://t.me/+M5HRCxZycqJmOTg5"
POLL_INTERVAL = 20          # seconds between Telegram polls
SESSION_FILE  = "unified_bot_session.txt"

# IBKR
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)
ORDER_SIZE     = 20

# Trading parameters
TREND_SCORE_ENTRY    = 85     # minimum trend score to enter
TREND_SCORE_WATCHLIST = 85    # minimum to remain on watchlist
TREND_SCORE_DROP_EXIT = 69    # trend score below this triggers soft stop
STOP_LOSS_1          = 0.10   # 10% soft stop (checks trend score)
STOP_LOSS_2          = 0.15   # 15% hard stop
TAKE_PROFIT_PCT      = 0.50   # 50% take profit
BASE_TRAILING_AMT    = 1.0    # $1 fixed trailing floor
MIN_TRAILING_PCT     = 0.005  # 0.5% trailing floor
ATR_MULTIPLIER       = 1.5
EOD_CLEAR_HOUR       = 16     # 4 PM — clear watchlist collection

# Entry scoring — need >= ENTRY_SCORE_MIN out of 3 conditions
ENTRY_SCORE_MIN      = 2

# PSAR parameters
PSAR_AF_START        = 0.02
PSAR_AF_INCREMENT    = 0.02
PSAR_AF_MAX          = 0.20


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB
# ══════════════════════════════════════════════════════════════════════════════

mongo_client = MongoClient("mongodb://localhost:27017/")
db           = mongo_client["breakout_db"]

watchlist_col  = db["watchlist_tickers"]    # ← shared ticker list (Telegram → Trading loop)
trades_col     = db["trades_v2"]
excluded_col   = db["excluded_tickers_v2"]

# Ensure unique index on 'symbol' so duplicate inserts are a no-op
watchlist_col.create_index("symbol", unique=True)


# ── Watchlist helpers ─────────────────────────────────────────────────────────

def watchlist_add(symbol: str, source: str = "telegram") -> bool:
    """
    Insert ticker into watchlist_tickers.
    Returns True if inserted, False if already existed (silently ignored).
    """
    try:
        watchlist_col.insert_one({
            "symbol":     symbol,
            "source":     source,
            "added_at":   datetime.now(timezone.utc),
            "active":     True,
        })
        log.info(f"WATCHLIST ADD  : {symbol}  (source={source})")
        return True
    except Exception:
        # Duplicate key — already in collection, skip silently
        log.debug(f"WATCHLIST DUP  : {symbol} already present, skipped.")
        return False


def watchlist_get_all() -> list[str]:
    """Return all active ticker symbols currently in the watchlist."""
    docs = watchlist_col.find({"active": True}, {"symbol": 1})
    return [d["symbol"] for d in docs]


def watchlist_clear_eod():
    """Delete ALL documents from watchlist_tickers at/after EOD hour."""
    now = datetime.now()
    if now.hour >= EOD_CLEAR_HOUR:
        result = watchlist_col.delete_many({})
        if result.deleted_count:
            log.info(f"EOD CLEAR: removed {result.deleted_count} ticker(s) from watchlist_tickers")


def watchlist_print():
    symbols = watchlist_get_all()
    if symbols:
        log.info(f"Watchlist ({len(symbols)}): {', '.join(symbols)}")
    else:
        log.info("Watchlist: empty")


# ══════════════════════════════════════════════════════════════════════════════
# TELEGRAM PARSERS
# ══════════════════════════════════════════════════════════════════════════════

def extract_ticker(text: str) -> str | None:
    text = text.strip()
    m = re.match(r'^\[.*?\]\s+([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    m = re.match(r'^([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    return None


def parse_structured_message(text: str) -> dict | None:
    result = {"symbol": None, "status": None}
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"(?i)^ticker\s*:", line):
            result["symbol"] = line.split(":", 1)[1].strip().upper()
        if re.match(r"(?i)^status\s*:", line):
            result["status"] = line.split(":", 1)[1].strip()
    if result["symbol"] and result["status"]:
        return result
    return None


def is_new_signal(status_text: str) -> bool:
    return "new" in status_text.lower()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

ib = IB()

def ibkr_connect():
    util.startLoop()
    ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
    log.info(f"IBKR connected | accounts: {ib.wrapper.accounts}")


# ── Gate helpers ──────────────────────────────────────────────────────────────

def has_open_position(symbol: str) -> bool:
    """Gate 1 — existing IB position check."""
    result = any(p.contract.symbol == symbol and p.position != 0 for p in ib.positions())
    log.debug(f"Gate 1 | has_open_position({symbol}) => {result}")
    return result


def has_pending_order(symbol: str) -> bool:
    """Gate 2 — pending IB order check."""
    result = any(t.contract.symbol == symbol for t in ib.openTrades())
    log.debug(f"Gate 2 | has_pending_order({symbol}) => {result}")
    return result


def get_ask_price(contract) -> float | None:
    """Gate 4 — fetch live ask / last / close from IB market data."""
    ticker = ib.reqMktData(contract, "106", False, False)
    ib.sleep(3)
    ib.cancelMktData(contract)
    for label, val in [("ask", ticker.ask), ("last", ticker.last), ("close", ticker.close)]:
        if val and float(val) > 0:
            log.info(f"Gate 4 | {contract.symbol}: source='{label}'  ${float(val):.4f}")
            return float(val)
    log.warning(f"Gate 4 | {contract.symbol}: all price fields empty")
    return None


# ══════════════════════════════════════════════════════════════════════════════
# INDICATOR CALCULATIONS
# ══════════════════════════════════════════════════════════════════════════════

def calculate_ema(df: pd.DataFrame, period: int) -> pd.Series:
    return df["close"].ewm(span=period, adjust=False).mean()


def calculate_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    high_low   = df["high"] - df["low"]
    high_close = np.abs(df["high"] - df["close"].shift())
    low_close  = np.abs(df["low"]  - df["close"].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["ATR"]  = true_range.rolling(period).mean()
    return df


def calculate_ema_200(df: pd.DataFrame) -> pd.DataFrame:
    df["ema_200"] = calculate_ema(df, 200)
    return df


def calculate_session_vwap(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date_only"] = pd.to_datetime(df["date"]).dt.date
    today           = df["date_only"].max()
    mask            = df["date_only"] == today
    today_df        = df.loc[mask].copy()
    today_df["vwap"] = (
        (today_df["close"] * today_df["volume"]).cumsum() /
        today_df["volume"].cumsum()
    )
    df.loc[mask, "vwap"] = today_df["vwap"].values
    df["vwap"] = df["vwap"].ffill()
    return df


# ── Parabolic SAR ────────────────────────────────────────────────────────────

def calculate_psar(df: pd.DataFrame,
                   af_start: float = PSAR_AF_START,
                   af_increment: float = PSAR_AF_INCREMENT,
                   af_max: float = PSAR_AF_MAX) -> pd.DataFrame:
    """
    Compute Parabolic SAR and add columns 'psar' and 'psar_bullish' to df.
    psar_bullish = True  when SAR is BELOW price (bullish / uptrend).
    psar_bullish = False when SAR is ABOVE price (bearish / downtrend).
    """
    high   = df["high"].values
    low    = df["low"].values
    close  = df["close"].values
    n      = len(df)

    psar          = np.zeros(n, dtype=float)
    psar_bullish  = np.zeros(n, dtype=bool)
    af            = af_start
    ep            = 0.0       # extreme point
    bull          = True

    # Initialise: first bar
    psar[0]         = low[0]
    psar_bullish[0] = True
    ep              = high[0]

    for i in range(1, n):
        # Compute tentative SAR
        psar[i] = psar[i - 1] + af * (ep - psar[i - 1])

        if bull:
            # In uptrend — SAR must not be above the two previous lows
            psar[i] = min(psar[i], low[i - 1])
            if i >= 2:
                psar[i] = min(psar[i], low[i - 2])

            if low[i] < psar[i]:
                # Reversal → switch to bearish
                bull    = False
                psar[i] = ep          # SAR flips to the extreme point
                ep      = low[i]
                af      = af_start
            else:
                if high[i] > ep:
                    ep = high[i]
                    af = min(af + af_increment, af_max)
        else:
            # In downtrend — SAR must not be below the two previous highs
            psar[i] = max(psar[i], high[i - 1])
            if i >= 2:
                psar[i] = max(psar[i], high[i - 2])

            if high[i] > psar[i]:
                # Reversal → switch to bullish
                bull    = True
                psar[i] = ep
                ep      = high[i]
                af      = af_start
            else:
                if low[i] < ep:
                    ep = low[i]
                    af = min(af + af_increment, af_max)

        psar_bullish[i] = bull

    df["psar"]         = psar
    df["psar_bullish"] = psar_bullish
    return df


# ── Volume Heatmap ───────────────────────────────────────────────────────────

def volume_heatmap(df: pd.DataFrame,
                   length: int = 60,
                   slength: int = 60,
                   threshold_extra_high: float = 4.0,
                   threshold_high: float = 2.5,
                   threshold_medium: float = 1.0,
                   threshold_normal: float = -0.5) -> pd.Series:
    """
    Classify each bar's volume relative to its rolling mean/std.
    Returns a Series of labels like 'Extra High Volume Up', 'Low Volume Down', etc.
    """
    mean   = df["volume"].rolling(window=length).mean()
    std    = df["volume"].rolling(window=slength).std()
    stdbar = (df["volume"] - mean) / std

    # Direction: did the bar close up?
    direction = df["close"] > df["open"]

    def classify(row):
        if pd.isna(row["stdbar"]):
            return "Insufficient Data"
        tag = " Up" if row["direction"] else " Down"
        if row["stdbar"] > threshold_extra_high:
            return "Extra High Volume" + tag
        elif row["stdbar"] > threshold_high:
            return "High Volume" + tag
        elif row["stdbar"] > threshold_medium:
            return "Medium Volume" + tag
        elif row["stdbar"] > threshold_normal:
            return "Normal Volume" + tag
        else:
            return "Low Volume" + tag

    temp = pd.DataFrame({"stdbar": stdbar, "direction": direction})
    return temp.apply(classify, axis=1)


# ── EMA 9 / 21 stack check ──────────────────────────────────────────────────

def calculate_ema_9_21(df: pd.DataFrame) -> pd.DataFrame:
    """Add ema_9, ema_21 and the bullish stack flag."""
    df["ema_9"]  = df["close"].ewm(span=9,  adjust=False).mean()
    df["ema_21"] = df["close"].ewm(span=21, adjust=False).mean()
    df["ema_9_21_bullish"] = (df["close"] > df["ema_9"]) & (df["ema_9"] > df["ema_21"])
    return df


# ── Combined entry score ─────────────────────────────────────────────────────

def compute_entry_score(row: pd.Series) -> tuple[int, dict]:
    """
    Evaluate the 3 entry conditions on the latest bar and return
    (total_score, detail_dict) where detail_dict maps condition name → bool.
    """
    cond_psar   = bool(row.get("psar_bullish", False))
    cond_volume = str(row.get("vol_heatmap", "")).startswith("Extra High Volume Up")
    cond_ema    = bool(row.get("ema_9_21_bullish", False))

    details = {
        "psar_bullish":        cond_psar,
        "extra_high_vol_up":   cond_volume,
        "ema_9_21_stack":      cond_ema,
    }
    score = int(cond_psar) + int(cond_volume) + int(cond_ema)
    return score, details


def calculate_ema_alignment_oscillator(df: pd.DataFrame,
                                       fast_period=8,
                                       mid_period=13,
                                       slow_period=14,
                                       trend_period=21,
                                       slope_threshold=1.0,
                                       spacing_min=0.2) -> pd.DataFrame:
    df["ema_fast"]  = calculate_ema(df, fast_period)
    df["ema_mid"]   = calculate_ema(df, mid_period)
    df["ema_slow"]  = calculate_ema(df, slow_period)
    df["ema_trend"] = calculate_ema(df, trend_period)
    df = calculate_atr(df, 14)

    def slope_deg(ema_s, atr_s):
        s = (ema_s - ema_s.shift(1)) / (atr_s + 0.0001) * 100
        return np.degrees(np.arctan(s))

    df["fast_angle"] = slope_deg(df["ema_fast"], df["ATR"])
    df["mid_angle"]  = slope_deg(df["ema_mid"],  df["ATR"])
    df["slow_angle"] = slope_deg(df["ema_slow"], df["ATR"])

    df["bullish_stack"] = (
        (df["ema_fast"]  > df["ema_mid"]) &
        (df["ema_mid"]   > df["ema_slow"]) &
        (df["ema_slow"]  > df["ema_trend"])
    )
    df["bearish_stack"] = (
        (df["ema_fast"]  < df["ema_mid"]) &
        (df["ema_mid"]   < df["ema_slow"]) &
        (df["ema_slow"]  < df["ema_trend"])
    )

    df["price_above_fast"] = df["close"] > df["ema_fast"]
    df["price_below_fast"] = df["close"] < df["ema_fast"]

    df["spacing_fast_mid"] = np.abs(df["ema_fast"] - df["ema_mid"]) / df["ema_mid"] * 100
    df["spacing_mid_slow"] = np.abs(df["ema_mid"]  - df["ema_slow"]) / df["ema_slow"] * 100
    df["avg_spacing"]      = (df["spacing_fast_mid"] + df["spacing_mid_slow"]) / 2

    df["volume_sma"]          = df["volume"].rolling(20).mean()
    df["volume_confirmation"] = df["volume"] > df["volume_sma"]

    df["trend_score"] = 0.0

    for i in range(1, len(df)):
        prev = df["trend_score"].iloc[i - 1]

        if df["bullish_stack"].iloc[i] and df["price_above_fast"].iloc[i]:
            avg_slope     = (df["fast_angle"].iloc[i] + df["mid_angle"].iloc[i] + df["slow_angle"].iloc[i]) / 3
            slope_bonus   = min(avg_slope * 3, 30)  if avg_slope > slope_threshold else 0
            spacing_bonus = min(df["avg_spacing"].iloc[i] * 10, 20) if df["avg_spacing"].iloc[i] > spacing_min else 0
            vol_bonus     = 10 if df["volume_confirmation"].iloc[i] else 0
            new_score     = 40.0 + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], "trend_score"] = (new_score + prev) / 2

        elif df["bearish_stack"].iloc[i] and df["price_below_fast"].iloc[i]:
            avg_slope     = (abs(df["fast_angle"].iloc[i]) + abs(df["mid_angle"].iloc[i]) + abs(df["slow_angle"].iloc[i])) / 3
            slope_bonus   = -min(avg_slope * 3, 30)  if avg_slope > slope_threshold else 0
            spacing_bonus = -min(df["avg_spacing"].iloc[i] * 10, 20) if df["avg_spacing"].iloc[i] > spacing_min else 0
            vol_bonus     = -10 if df["volume_confirmation"].iloc[i] else 0
            new_score     = -40.0 + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], "trend_score"] = (new_score + prev) / 2

        else:
            df.loc[df.index[i], "trend_score"] = prev * 0.9

    df.loc[~df["volume_confirmation"], "trend_score"] *= 0.8
    df["trend_score"] = df["trend_score"].clip(-100, 100)

    # Threshold = 85
    df["strong_bull_signal"] = (
        (df["trend_score"] >= TREND_SCORE_ENTRY) &
        (df["trend_score"] > df["trend_score"].shift(1))
    )
    df["strong_bear_signal"] = (
        (df["trend_score"] <= -TREND_SCORE_ENTRY) &
        (df["trend_score"] < df["trend_score"].shift(1))
    )
    return df


# ══════════════════════════════════════════════════════════════════════════════
# TRADE DOCUMENT HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_document(symbol, entry_price, quantity, trend_score_at_entry,
                          entry_score=0, entry_details=None):
    return {
        "instrument":        symbol,
        "direction":         "long",
        "entry_price":       entry_price,
        "highest_price":     entry_price,
        "quantity":          quantity,
        "entry_timestamp":   datetime.now(),
        "entry_trend_score": trend_score_at_entry,
        "entry_score":       entry_score,
        "entry_details":     entry_details or {},
        "order_id":          str(ObjectId()),
        "exit_price":        None,
        "exit_timestamp":    None,
        "reason":            None,
        "current_pnl":       0,
        "realized_pnl":      0,
        "status":            "open",
        "created_at":        datetime.now(),
        "updated_at":        datetime.now(),
    }


def update_trade_document(trade_id, updates: dict):
    updates["updated_at"] = datetime.now()
    trades_col.update_one({"_id": ObjectId(trade_id)}, {"$set": updates})


# ══════════════════════════════════════════════════════════════════════════════
# TRADING LOOP  (runs as asyncio task every 5 minutes)
# ══════════════════════════════════════════════════════════════════════════════

async def trading_loop():
    log.info("Trading loop started.")
    await asyncio.sleep(10)   # brief startup grace period

    while True:
        # ── EOD cleanup ───────────────────────────────────────────────────────
        watchlist_clear_eod()
        watchlist_print()

        # ── Load tickers dynamically from MongoDB ─────────────────────────────
        symbols = watchlist_get_all()

        if not symbols:
            log.info("Watchlist empty — nothing to evaluate. Waiting 5 min...")
            await asyncio.sleep(300)
            continue

        log.info(f"Evaluating {len(symbols)} ticker(s): {', '.join(symbols)}")

        # Qualify contracts fresh each cycle (watchlist may have changed)
        contracts = [Stock(s, "SMART", "USD") for s in symbols]
        try:
            ib.qualifyContracts(*contracts)
        except Exception as e:
            log.warning(f"qualifyContracts error: {e}")

        for contract in contracts:
            symbol = contract.symbol
            log.info(f"\n{'='*60}")
            log.info(f"Evaluating {symbol} at {datetime.now().strftime('%H:%M:%S')}")
            log.info(f"{'='*60}")

            # ── Fetch 5-min bars ──────────────────────────────────────────────
            try:
                bars = ib.reqHistoricalData(
                    contract,
                    endDateTime="",
                    durationStr="2 D",
                    barSizeSetting="5 mins",
                    whatToShow="TRADES",
                    useRTH=False,
                    formatDate=1,
                )
            except Exception as e:
                log.warning(f"{symbol}: reqHistoricalData error — {e}")
                continue

            if not bars:
                log.warning(f"{symbol}: no historical data returned")
                continue

            df = util.df(bars)
            df.columns = [c.lower() for c in df.columns]

            # ── Calculate ALL indicators ──────────────────────────────────────
            df = calculate_ema_alignment_oscillator(df)
            df = calculate_ema_200(df)
            df = calculate_session_vwap(df)
            df = calculate_psar(df)                       # NEW: Parabolic SAR
            df["vol_heatmap"] = volume_heatmap(df)        # NEW: Volume Heatmap
            df = calculate_ema_9_21(df)                   # NEW: EMA 9/21 stack

            row           = df.iloc[-1]
            current_price = row["close"]
            trend_score   = row["trend_score"]
            atr           = row.get("ATR", BASE_TRAILING_AMT)
            vwap          = row["vwap"]

            # Compute entry score for logging (even when managing an open trade)
            entry_score, entry_details = compute_entry_score(row)

            log.info(f"Price: ${current_price:.2f} | TrendScore: {trend_score:.1f} | "
                     f"EMA200: ${row['ema_200']:.2f} | VWAP: ${vwap:.2f}")
            log.info(f"EntryScore: {entry_score}/3 | "
                     f"PSAR={'✓' if entry_details['psar_bullish'] else '✗'} | "
                     f"VolHeat={'✓' if entry_details['extra_high_vol_up'] else '✗'} ({row['vol_heatmap']}) | "
                     f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '✗'}")

            # ── Check for open MongoDB trade ──────────────────────────────────
            open_trade = trades_col.find_one({"instrument": symbol, "status": "open"})

            # ═════════════════════════════════════════════════════════════════
            # SELL / POSITION MANAGEMENT  (unchanged)
            # ═════════════════════════════════════════════════════════════════

            if open_trade:
                entry_price   = open_trade["entry_price"]
                highest_price = open_trade.get("highest_price", entry_price)
                quantity      = open_trade["quantity"]
                time_in_trade = (datetime.now() - open_trade["entry_timestamp"]).total_seconds() / 3600

                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade["_id"], {"highest_price": highest_price})

                profit_pct  = (current_price - entry_price) / entry_price
                pnl_ps      = current_price - entry_price
                current_pnl = pnl_ps * quantity
                update_trade_document(open_trade["_id"], {"current_pnl": current_pnl})

                log.info(f"OPEN | Entry: ${entry_price:.2f} | P&L: {profit_pct:.2%} | Score: {trend_score:.1f}")

                # ── Stop loss ─────────────────────────────────────────────────
                should_exit = False
                exit_reason = None

                if profit_pct <= -STOP_LOSS_1:
                    if trend_score < TREND_SCORE_DROP_EXIT:
                        should_exit = True
                        exit_reason = "stop_loss_10pct_trend_weak"
                        log.info(f"EXIT: 10% loss + score {trend_score:.1f} < {TREND_SCORE_DROP_EXIT}")
                    elif profit_pct <= -STOP_LOSS_2:
                        should_exit = True
                        exit_reason = "stop_loss_15pct_hard"
                        log.info(f"EXIT: 15% hard stop reached")
                    else:
                        log.info(f"Holding at {profit_pct:.2%} loss — trend still strong ({trend_score:.1f})")

                if should_exit and quantity > 0:
                    ib.placeOrder(contract, MarketOrder("SELL", quantity))
                    realized = pnl_ps * quantity
                    update_trade_document(open_trade["_id"], {
                        "exit_price":       current_price,
                        "exit_timestamp":   datetime.now(),
                        "reason":           exit_reason,
                        "realized_pnl":     realized,
                        "exit_trend_score": trend_score,
                        "status":           "closed",
                    })
                    log.info(f"STOP LOSS | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                    await asyncio.sleep(0.5)
                    continue

                # ── Take profit ───────────────────────────────────────────────
                if profit_pct >= TAKE_PROFIT_PCT:
                    ib.placeOrder(contract, MarketOrder("SELL", quantity))
                    realized = pnl_ps * quantity
                    update_trade_document(open_trade["_id"], {
                        "exit_price":       current_price,
                        "exit_timestamp":   datetime.now(),
                        "reason":           f"take_profit_{profit_pct:.2%}",
                        "realized_pnl":     realized,
                        "exit_trend_score": trend_score,
                        "status":           "closed",
                    })
                    log.info(f"TAKE PROFIT | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                    await asyncio.sleep(0.5)
                    continue

                # ── Trailing stop (only when profitable) ──────────────────────
                if profit_pct > 0:
                    atr_stop  = highest_price - (atr * ATR_MULTIPLIER)
                    pct_stop  = highest_price * (1 - MIN_TRAILING_PCT)
                    fix_stop  = highest_price - BASE_TRAILING_AMT
                    t_factor  = min(1.0, time_in_trade / 24)
                    trail_stop = max(atr_stop * (1 - 0.25 * t_factor), pct_stop, fix_stop)

                    if current_price <= trail_stop and quantity > 0:
                        ib.placeOrder(contract, MarketOrder("SELL", quantity))
                        realized = pnl_ps * quantity
                        update_trade_document(open_trade["_id"], {
                            "exit_price":       current_price,
                            "exit_timestamp":   datetime.now(),
                            "reason":           "trailing_stop_hit",
                            "realized_pnl":     realized,
                            "exit_trend_score": trend_score,
                            "status":           "closed",
                        })
                        log.info(f"TRAILING STOP | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                        await asyncio.sleep(0.5)
                        continue

            # ═════════════════════════════════════════════════════════════════
            # BUY / ENTRY LOGIC  (v2 — scoring-based)
            #
            # Scoring system (1 point each, need >= 2 to enter):
            #   1. PSAR bullish  (SAR below price)
            #   2. Volume Heatmap = "Extra High Volume Up"
            #   3. Close > EMA 9 > EMA 21
            #
            # Gate checks (position / order / price) still apply after
            # the score threshold is met.
            # ═════════════════════════════════════════════════════════════════

            else:
                # ── Already bought this ticker today? Skip entirely. ───────────
                today_str     = datetime.now().date().isoformat()
                exclude_entry = excluded_col.find_one({"ticker": symbol, "date": today_str})
                if exclude_entry:
                    log.info(f"{symbol}: already bought today — skipping entry evaluation")
                    await asyncio.sleep(0.5)
                    continue

                # ── Entry score check (replaces old bull_signal + EMA200 + VWAP) ─
                if entry_score < ENTRY_SCORE_MIN:
                    log.info(
                        f"{symbol}: entry score {entry_score}/3 < {ENTRY_SCORE_MIN} — "
                        f"PSAR={'✓' if entry_details['psar_bullish'] else '✗'} | "
                        f"VolHeat={'✓' if entry_details['extra_high_vol_up'] else '✗'} | "
                        f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '✗'} "
                        f"— will re-check next cycle"
                    )
                    await asyncio.sleep(0.5)
                    continue

                log.info(
                    f"{symbol}: ENTRY SCORE {entry_score}/3 ≥ {ENTRY_SCORE_MIN} ✓  "
                    f"PSAR={'✓' if entry_details['psar_bullish'] else '✗'} | "
                    f"VolHeat={'✓' if entry_details['extra_high_vol_up'] else '✗'} | "
                    f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '✗'}"
                )

                # ── Gate 1: no existing IB position ──────────────────────────
                if has_open_position(symbol):
                    log.info(f"{symbol}: Gate 1 FAILED — existing IB position — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 2: no pending IB order ───────────────────────────────
                if has_pending_order(symbol):
                    log.info(f"{symbol}: Gate 2 FAILED — pending IB order — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 4: valid ask price ───────────────────────────────────
                ask_price = get_ask_price(contract)
                if not ask_price:
                    log.info(f"{symbol}: Gate 4 FAILED — no valid ask price — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue

                # ── All gates passed → place order ────────────────────────────
                ib.placeOrder(contract, MarketOrder("BUY", ORDER_SIZE))

                trade_doc = create_trade_document(
                    symbol               = symbol,
                    entry_price          = ask_price,
                    quantity             = ORDER_SIZE,
                    trend_score_at_entry = trend_score,
                    entry_score          = entry_score,
                    entry_details        = entry_details,
                )
                trades_col.insert_one(trade_doc)

                # ── Mark as bought ONLY here, after a successful order ─────────
                excluded_col.insert_one({"ticker": symbol, "date": today_str})

                log.info(
                    f"BUY | {symbol} | ask=${ask_price:.2f} | qty={ORDER_SIZE} | "
                    f"score={entry_score}/3 | trend={trend_score:.1f} | "
                    f"PSAR ✓ | VolHeat={'✓' if entry_details['extra_high_vol_up'] else '—'} | "
                    f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '—'} | "
                    f"G1 ✓ | G2 ✓ | G4 ✓"
                )

            await asyncio.sleep(0.5)   # brief pause between tickers

        # ── End of scan cycle ─────────────────────────────────────────────────
        log.info(f"\n{'='*60}")
        log.info("Scan complete. Next scan in 5 minutes.")
        watchlist_print()
        log.info(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# TELEGRAM POLLER  (runs as asyncio task continuously)
# Adds new tickers to MongoDB watchlist_tickers collection.
# ══════════════════════════════════════════════════════════════════════════════

def load_session() -> str:
    if os.path.exists(SESSION_FILE):
        with open(SESSION_FILE, encoding="utf-8") as f:
            return f.read().strip()
    return ""


def save_session(s: str):
    with open(SESSION_FILE, "w", encoding="utf-8") as f:
        f.write(s)
    log.info(f"Telegram session saved → {SESSION_FILE}")


async def telegram_poller():
    log.info("Telegram poller started.")

    while True:
        tg = TelegramClient(StringSession(load_session()), API_ID, API_HASH)
        try:
            await tg.start(phone=PHONE)
            save_session(tg.session.save())

            # Join channel if needed
            m = re.search(r't\.me/(?:joinchat/|\+)([A-Za-z0-9_-]+)', INVITE_LINK)
            if not m:
                raise ValueError(f"Cannot parse invite hash: {INVITE_LINK}")
            try:
                await tg(ImportChatInviteRequest(m.group(1)))
                log.info("Joined Telegram channel")
            except UserAlreadyParticipantError:
                log.info("Already a member of the channel")

            channel      = await tg.get_entity(INVITE_LINK)
            last_seen_id = None
            log.info(f"Polling channel: {channel.title} (ID: {channel.id})")

            while True:
                messages = await tg.get_messages(channel, limit=1)
                if messages:
                    msg    = messages[0]
                    is_new = (last_seen_id is None or msg.id != last_seen_id)

                    if msg.text and is_new:
                        parsed = parse_structured_message(msg.text)
                        if parsed:
                            symbol = parsed["symbol"]
                            status = parsed["status"]
                        else:
                            symbol = extract_ticker(msg.text)
                            status = "NEW"

                        preview = msg.text[:120] + ("..." if len(msg.text) > 120 else "")
                        safe_print(f"\n[{msg.date}]")
                        safe_print(f"   Message : {preview}")
                        safe_print(f"   Ticker  : {symbol or '(not found)'}")
                        safe_print(f"   Status  : NEW")

                        if symbol and is_new_signal(status):
                            inserted = watchlist_add(symbol, source="telegram")
                            if inserted:
                                safe_print(f"   → Added {symbol} to watchlist_tickers in MongoDB")
                            else:
                                safe_print(f"   → {symbol} already in watchlist_tickers, skipped")
                        elif symbol:
                            log.info(f"{symbol}: status='{status}' — not NEW, skipping watchlist add")

                        last_seen_id = msg.id

                log.debug(f"Telegram: waiting {POLL_INTERVAL}s...")
                await asyncio.sleep(POLL_INTERVAL)

        except FloodWaitError as e:
            log.warning(f"Telegram rate-limited — waiting {e.seconds}s")
            await asyncio.sleep(e.seconds + 5)
        except KeyboardInterrupt:
            log.info("Telegram poller stopped by user.")
            break
        except Exception as e:
            log.error(f"Telegram error: {type(e).__name__}: {e} — reconnecting in 15s")
            await asyncio.sleep(15)
        finally:
            try:
                await tg.disconnect()
            except Exception:
                pass


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT — run both tasks concurrently on the same event loop
# ══════════════════════════════════════════════════════════════════════════════

async def main():
    log.info("=== Unified Trading Bot Starting ===")
    ibkr_connect()
    await asyncio.gather(
        telegram_poller(),   # continuously polls Telegram & updates watchlist_tickers
        trading_loop(),      # every 5 min reads watchlist_tickers & manages trades
    )


if __name__ == "__main__":
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        log.info("Bot stopped by user.")
    finally:
        ib.disconnect()
        log.info("IBKR disconnected.")


In [ ]:
"""
Unified Trading Bot
═══════════════════
Telegram Poller  ──►  MongoDB (watchlist_tickers)  ──►  IBKR Trading Loop

Flow:
  1. Telethon polls a private Telegram channel every 20 seconds.
  2. NEW signals are parsed for a ticker symbol and upserted into
     the 'watchlist_tickers' MongoDB collection.
  3. The IBKR trading loop runs every 5 minutes, reads ALL active
     tickers from 'watchlist_tickers', qualifies contracts on-the-fly,
     and evaluates buy / position-management logic for each.
  4. At EOD (>= 4 PM) the watchlist_tickers collection is cleared.

ENTRY SCORING (v2):
  Each ticker is scored on 3 conditions (1 point each):
    1. Parabolic SAR indicates bullish entry (SAR below price)
    2. Volume Heatmap shows "Extra High Volume Up"
    3. Close > EMA 9 > EMA 21  (bullish EMA stack)
  Score >= 2  →  enter position

ORDER ROUTING:
  - Regular Trading Hours (9:30–16:00 ET): MarketOrder
  - Premarket / After-hours: LimitOrder at ask price + 1% slippage,
    outsideRth=True, tif=DAY
  This prevents IBKR error 10349 (order cancelled due to TIF/GTC preset
  conflicts with market orders outside RTH).

EVENT LOOP:
  Both Telethon and ib_insync share the SAME asyncio event loop via
  nest_asyncio. The Telegram poller and the trading loop run as
  concurrent asyncio tasks — no threads needed.
"""

import asyncio
import os
import re
import sys
import random
import logging
import warnings
from datetime import datetime, timezone, timedelta

import nest_asyncio
nest_asyncio.apply()          # must be before any IB / asyncio.run() usage

import numpy as np
import pandas as pd
from pymongo import MongoClient
from bson import ObjectId

from ib_insync import IB, Stock, MarketOrder, LimitOrder, util

from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import FloodWaitError, UserAlreadyParticipantError
from telethon.tl.functions.messages import ImportChatInviteRequest

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt    = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh     = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh     = logging.FileHandler("unified_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("unified_bot")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


def safe_print(text: str):
    try:
        encoded = text.encode(sys.stdout.encoding or "utf-8", errors="replace")
        sys.stdout.buffer.write(encoded + b"\n")
        sys.stdout.buffer.flush()
    except Exception:
        print(text)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

# Telegram
API_ID        = 34025130
API_HASH      = "e5e514ebcb8a15042feadfecd7a15cfe"
PHONE         = "+18477675748"
INVITE_LINK   = "https://t.me/+M5HRCxZycqJmOTg5"
POLL_INTERVAL = 20          # seconds between Telegram polls
SESSION_FILE  = "unified_bot_session.txt"

# IBKR
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)
ORDER_SIZE     = 20

# Trading parameters
TREND_SCORE_ENTRY    = 85     # minimum trend score to enter
TREND_SCORE_WATCHLIST = 85    # minimum to remain on watchlist
TREND_SCORE_DROP_EXIT = 69    # trend score below this triggers soft stop
STOP_LOSS_1          = 0.10   # 10% soft stop (checks trend score)
STOP_LOSS_2          = 0.15   # 15% hard stop
TAKE_PROFIT_PCT      = 0.50   # 50% take profit
BASE_TRAILING_AMT    = 1.0    # $1 fixed trailing floor
MIN_TRAILING_PCT     = 0.005  # 0.5% trailing floor
ATR_MULTIPLIER       = 1.5
EOD_CLEAR_HOUR       = 16     # 4 PM — clear watchlist collection

# Entry scoring — need >= ENTRY_SCORE_MIN out of 3 conditions
ENTRY_SCORE_MIN      = 2

# PSAR parameters
PSAR_AF_START        = 0.02
PSAR_AF_INCREMENT    = 0.02
PSAR_AF_MAX          = 0.20

# RTH hours (Eastern Time)
RTH_START_HOUR       = 9
RTH_START_MINUTE     = 30
RTH_END_HOUR         = 16
RTH_END_MINUTE       = 0

# Limit order settings for extended hours
LIMIT_PRICE_SLIPPAGE = 0.01   # 1% above ask for BUY, 1% below bid for SELL


# ══════════════════════════════════════════════════════════════════════════════
# ORDER ROUTING — RTH vs Extended Hours
# ══════════════════════════════════════════════════════════════════════════════

def _get_eastern_now() -> datetime:
    """
    Get current time in US Eastern.
    Uses a fixed UTC-5 / UTC-4 offset based on month (simple DST heuristic).
    For production, consider using pytz or zoneinfo.
    """
    utc_now = datetime.now(timezone.utc)
    month = utc_now.month
    if 3 <= month <= 10:
        offset = timedelta(hours=-4)   # EDT
    else:
        offset = timedelta(hours=-5)   # EST
    return utc_now + offset


def is_rth() -> bool:
    """Check if we're inside Regular Trading Hours (9:30–16:00 ET)."""
    et_now = _get_eastern_now()
    rth_open  = et_now.replace(hour=RTH_START_HOUR, minute=RTH_START_MINUTE, second=0, microsecond=0)
    rth_close = et_now.replace(hour=RTH_END_HOUR,   minute=RTH_END_MINUTE,   second=0, microsecond=0)
    return rth_open <= et_now < rth_close


def create_buy_order(qty: int, limit_price: float = None):
    """
    Create the appropriate buy order based on market hours.
    - RTH:      MarketOrder (fastest fill)
    - Extended:  LimitOrder at (ask + slippage), outsideRth=True
    """
    if is_rth():
        log.info(f"ORDER: MarketOrder BUY x{qty} (RTH)")
        return MarketOrder("BUY", qty)
    else:
        if limit_price is None:
            raise ValueError("limit_price required for extended-hours BUY order")
        aggressive_price = round(limit_price * (1 + LIMIT_PRICE_SLIPPAGE), 2)
        order = LimitOrder("BUY", qty, aggressive_price)
        order.outsideRth = True
        order.tif = "DAY"
        log.info(f"ORDER: LimitOrder BUY x{qty} @ ${aggressive_price:.2f} "
                 f"(extended hours, ask=${limit_price:.2f} + {LIMIT_PRICE_SLIPPAGE*100:.0f}% slip)")
        return order


def create_sell_order(qty: int, limit_price: float = None):
    """
    Create the appropriate sell order based on market hours.
    - RTH:      MarketOrder (fastest fill)
    - Extended:  LimitOrder at (bid - slippage), outsideRth=True
    """
    if is_rth():
        log.info(f"ORDER: MarketOrder SELL x{qty} (RTH)")
        return MarketOrder("SELL", qty)
    else:
        if limit_price is None:
            raise ValueError("limit_price required for extended-hours SELL order")
        aggressive_price = round(limit_price * (1 - LIMIT_PRICE_SLIPPAGE), 2)
        order = LimitOrder("SELL", qty, aggressive_price)
        order.outsideRth = True
        order.tif = "DAY"
        log.info(f"ORDER: LimitOrder SELL x{qty} @ ${aggressive_price:.2f} "
                 f"(extended hours, price=${limit_price:.2f} - {LIMIT_PRICE_SLIPPAGE*100:.0f}% slip)")
        return order


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB
# ══════════════════════════════════════════════════════════════════════════════

mongo_client = MongoClient("mongodb://localhost:27017/")
db           = mongo_client["breakout_db"]

watchlist_col  = db["watchlist_tickers"]
trades_col     = db["trades_v2"]
excluded_col   = db["excluded_tickers_v2"]

watchlist_col.create_index("symbol", unique=True)


# ── Watchlist helpers ─────────────────────────────────────────────────────────

def watchlist_add(symbol: str, source: str = "telegram") -> bool:
    try:
        watchlist_col.insert_one({
            "symbol":     symbol,
            "source":     source,
            "added_at":   datetime.now(timezone.utc),
            "active":     True,
        })
        log.info(f"WATCHLIST ADD  : {symbol}  (source={source})")
        return True
    except Exception:
        log.debug(f"WATCHLIST DUP  : {symbol} already present, skipped.")
        return False


def watchlist_get_all() -> list[str]:
    docs = watchlist_col.find({"active": True}, {"symbol": 1})
    return [d["symbol"] for d in docs]


def watchlist_clear_eod():
    now = datetime.now()
    if now.hour >= EOD_CLEAR_HOUR:
        result = watchlist_col.delete_many({})
        if result.deleted_count:
            log.info(f"EOD CLEAR: removed {result.deleted_count} ticker(s) from watchlist_tickers")


def watchlist_print():
    symbols = watchlist_get_all()
    if symbols:
        log.info(f"Watchlist ({len(symbols)}): {', '.join(symbols)}")
    else:
        log.info("Watchlist: empty")


# ══════════════════════════════════════════════════════════════════════════════
# TELEGRAM PARSERS
# ══════════════════════════════════════════════════════════════════════════════

def extract_ticker(text: str) -> str | None:
    text = text.strip()
    m = re.match(r'^\[.*?\]\s+([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    m = re.match(r'^([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    return None


def parse_structured_message(text: str) -> dict | None:
    result = {"symbol": None, "status": None}
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"(?i)^ticker\s*:", line):
            result["symbol"] = line.split(":", 1)[1].strip().upper()
        if re.match(r"(?i)^status\s*:", line):
            result["status"] = line.split(":", 1)[1].strip()
    if result["symbol"] and result["status"]:
        return result
    return None


def is_new_signal(status_text: str) -> bool:
    return "new" in status_text.lower()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

ib = IB()

def ibkr_connect():
    util.startLoop()
    ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
    log.info(f"IBKR connected | accounts: {ib.wrapper.accounts}")


# ── Gate helpers ──────────────────────────────────────────────────────────────

def has_open_position(symbol: str) -> bool:
    result = any(p.contract.symbol == symbol and p.position != 0 for p in ib.positions())
    log.debug(f"Gate 1 | has_open_position({symbol}) => {result}")
    return result


def has_pending_order(symbol: str) -> bool:
    result = any(t.contract.symbol == symbol for t in ib.openTrades())
    log.debug(f"Gate 2 | has_pending_order({symbol}) => {result}")
    return result


def get_ask_price(contract) -> float | None:
    """Gate 4 — fetch live ask / last / close from IB market data."""
    ticker = ib.reqMktData(contract, "106", False, False)
    ib.sleep(3)
    ib.cancelMktData(contract)
    for label, val in [("ask", ticker.ask), ("last", ticker.last), ("close", ticker.close)]:
        if val and float(val) > 0:
            log.info(f"Gate 4 | {contract.symbol}: source='{label}'  ${float(val):.4f}")
            return float(val)
    log.warning(f"Gate 4 | {contract.symbol}: all price fields empty")
    return None


def get_bid_price(contract) -> float | None:
    """Fetch live bid / last / close — used for sell-side limit pricing."""
    ticker = ib.reqMktData(contract, "106", False, False)
    ib.sleep(3)
    ib.cancelMktData(contract)
    for label, val in [("bid", ticker.bid), ("last", ticker.last), ("close", ticker.close)]:
        if val and float(val) > 0:
            log.info(f"Bid | {contract.symbol}: source='{label}'  ${float(val):.4f}")
            return float(val)
    log.warning(f"Bid | {contract.symbol}: all price fields empty")
    return None


# ══════════════════════════════════════════════════════════════════════════════
# INDICATOR CALCULATIONS
# ══════════════════════════════════════════════════════════════════════════════

def calculate_ema(df: pd.DataFrame, period: int) -> pd.Series:
    return df["close"].ewm(span=period, adjust=False).mean()


def calculate_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    high_low   = df["high"] - df["low"]
    high_close = np.abs(df["high"] - df["close"].shift())
    low_close  = np.abs(df["low"]  - df["close"].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["ATR"]  = true_range.rolling(period).mean()
    return df


def calculate_ema_200(df: pd.DataFrame) -> pd.DataFrame:
    df["ema_200"] = calculate_ema(df, 200)
    return df


def calculate_session_vwap(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date_only"] = pd.to_datetime(df["date"]).dt.date
    today           = df["date_only"].max()
    mask            = df["date_only"] == today
    today_df        = df.loc[mask].copy()
    today_df["vwap"] = (
        (today_df["close"] * today_df["volume"]).cumsum() /
        today_df["volume"].cumsum()
    )
    df.loc[mask, "vwap"] = today_df["vwap"].values
    df["vwap"] = df["vwap"].ffill()
    return df


# ── Parabolic SAR ────────────────────────────────────────────────────────────

def calculate_psar(df: pd.DataFrame,
                   af_start: float = PSAR_AF_START,
                   af_increment: float = PSAR_AF_INCREMENT,
                   af_max: float = PSAR_AF_MAX) -> pd.DataFrame:
    high   = df["high"].values
    low    = df["low"].values
    close  = df["close"].values
    n      = len(df)

    psar          = np.zeros(n, dtype=float)
    psar_bullish  = np.zeros(n, dtype=bool)
    af            = af_start
    ep            = 0.0
    bull          = True

    psar[0]         = low[0]
    psar_bullish[0] = True
    ep              = high[0]

    for i in range(1, n):
        psar[i] = psar[i - 1] + af * (ep - psar[i - 1])

        if bull:
            psar[i] = min(psar[i], low[i - 1])
            if i >= 2:
                psar[i] = min(psar[i], low[i - 2])
            if low[i] < psar[i]:
                bull    = False
                psar[i] = ep
                ep      = low[i]
                af      = af_start
            else:
                if high[i] > ep:
                    ep = high[i]
                    af = min(af + af_increment, af_max)
        else:
            psar[i] = max(psar[i], high[i - 1])
            if i >= 2:
                psar[i] = max(psar[i], high[i - 2])
            if high[i] > psar[i]:
                bull    = True
                psar[i] = ep
                ep      = high[i]
                af      = af_start
            else:
                if low[i] < ep:
                    ep = low[i]
                    af = min(af + af_increment, af_max)

        psar_bullish[i] = bull

    df["psar"]         = psar
    df["psar_bullish"] = psar_bullish
    return df


# ── Volume Heatmap ───────────────────────────────────────────────────────────

def volume_heatmap(df: pd.DataFrame,
                   length: int = 60,
                   slength: int = 60,
                   threshold_extra_high: float = 4.0,
                   threshold_high: float = 2.5,
                   threshold_medium: float = 1.0,
                   threshold_normal: float = -0.5) -> pd.Series:
    mean   = df["volume"].rolling(window=length).mean()
    std    = df["volume"].rolling(window=slength).std()
    stdbar = (df["volume"] - mean) / std
    direction = df["close"] > df["open"]

    def classify(row):
        if pd.isna(row["stdbar"]):
            return "Insufficient Data"
        tag = " Up" if row["direction"] else " Down"
        if row["stdbar"] > threshold_extra_high:
            return "Extra High Volume" + tag
        elif row["stdbar"] > threshold_high:
            return "High Volume" + tag
        elif row["stdbar"] > threshold_medium:
            return "Medium Volume" + tag
        elif row["stdbar"] > threshold_normal:
            return "Normal Volume" + tag
        else:
            return "Low Volume" + tag

    temp = pd.DataFrame({"stdbar": stdbar, "direction": direction})
    return temp.apply(classify, axis=1)


# ── EMA 9 / 21 stack check ──────────────────────────────────────────────────

def calculate_ema_9_21(df: pd.DataFrame) -> pd.DataFrame:
    df["ema_9"]  = df["close"].ewm(span=9,  adjust=False).mean()
    df["ema_21"] = df["close"].ewm(span=21, adjust=False).mean()
    df["ema_9_21_bullish"] = (df["close"] > df["ema_9"]) & (df["ema_9"] > df["ema_21"])
    return df


# ── Combined entry score ─────────────────────────────────────────────────────

def compute_entry_score(row: pd.Series) -> tuple[int, dict]:
    cond_psar   = bool(row.get("psar_bullish", False))
    cond_volume = str(row.get("vol_heatmap", "")).startswith("Extra High Volume Up")
    cond_ema    = bool(row.get("ema_9_21_bullish", False))

    details = {
        "psar_bullish":        cond_psar,
        "extra_high_vol_up":   cond_volume,
        "ema_9_21_stack":      cond_ema,
    }
    score = int(cond_psar) + int(cond_volume) + int(cond_ema)
    return score, details


def calculate_ema_alignment_oscillator(df: pd.DataFrame,
                                       fast_period=8,
                                       mid_period=13,
                                       slow_period=14,
                                       trend_period=21,
                                       slope_threshold=1.0,
                                       spacing_min=0.2) -> pd.DataFrame:
    df["ema_fast"]  = calculate_ema(df, fast_period)
    df["ema_mid"]   = calculate_ema(df, mid_period)
    df["ema_slow"]  = calculate_ema(df, slow_period)
    df["ema_trend"] = calculate_ema(df, trend_period)
    df = calculate_atr(df, 14)

    def slope_deg(ema_s, atr_s):
        s = (ema_s - ema_s.shift(1)) / (atr_s + 0.0001) * 100
        return np.degrees(np.arctan(s))

    df["fast_angle"] = slope_deg(df["ema_fast"], df["ATR"])
    df["mid_angle"]  = slope_deg(df["ema_mid"],  df["ATR"])
    df["slow_angle"] = slope_deg(df["ema_slow"], df["ATR"])

    df["bullish_stack"] = (
        (df["ema_fast"]  > df["ema_mid"]) &
        (df["ema_mid"]   > df["ema_slow"]) &
        (df["ema_slow"]  > df["ema_trend"])
    )
    df["bearish_stack"] = (
        (df["ema_fast"]  < df["ema_mid"]) &
        (df["ema_mid"]   < df["ema_slow"]) &
        (df["ema_slow"]  < df["ema_trend"])
    )

    df["price_above_fast"] = df["close"] > df["ema_fast"]
    df["price_below_fast"] = df["close"] < df["ema_fast"]

    df["spacing_fast_mid"] = np.abs(df["ema_fast"] - df["ema_mid"]) / df["ema_mid"] * 100
    df["spacing_mid_slow"] = np.abs(df["ema_mid"]  - df["ema_slow"]) / df["ema_slow"] * 100
    df["avg_spacing"]      = (df["spacing_fast_mid"] + df["spacing_mid_slow"]) / 2

    df["volume_sma"]          = df["volume"].rolling(20).mean()
    df["volume_confirmation"] = df["volume"] > df["volume_sma"]

    df["trend_score"] = 0.0

    for i in range(1, len(df)):
        prev = df["trend_score"].iloc[i - 1]

        if df["bullish_stack"].iloc[i] and df["price_above_fast"].iloc[i]:
            avg_slope     = (df["fast_angle"].iloc[i] + df["mid_angle"].iloc[i] + df["slow_angle"].iloc[i]) / 3
            slope_bonus   = min(avg_slope * 3, 30)  if avg_slope > slope_threshold else 0
            spacing_bonus = min(df["avg_spacing"].iloc[i] * 10, 20) if df["avg_spacing"].iloc[i] > spacing_min else 0
            vol_bonus     = 10 if df["volume_confirmation"].iloc[i] else 0
            new_score     = 40.0 + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], "trend_score"] = (new_score + prev) / 2

        elif df["bearish_stack"].iloc[i] and df["price_below_fast"].iloc[i]:
            avg_slope     = (abs(df["fast_angle"].iloc[i]) + abs(df["mid_angle"].iloc[i]) + abs(df["slow_angle"].iloc[i])) / 3
            slope_bonus   = -min(avg_slope * 3, 30)  if avg_slope > slope_threshold else 0
            spacing_bonus = -min(df["avg_spacing"].iloc[i] * 10, 20) if df["avg_spacing"].iloc[i] > spacing_min else 0
            vol_bonus     = -10 if df["volume_confirmation"].iloc[i] else 0
            new_score     = -40.0 + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], "trend_score"] = (new_score + prev) / 2

        else:
            df.loc[df.index[i], "trend_score"] = prev * 0.9

    df.loc[~df["volume_confirmation"], "trend_score"] *= 0.8
    df["trend_score"] = df["trend_score"].clip(-100, 100)

    df["strong_bull_signal"] = (
        (df["trend_score"] >= TREND_SCORE_ENTRY) &
        (df["trend_score"] > df["trend_score"].shift(1))
    )
    df["strong_bear_signal"] = (
        (df["trend_score"] <= -TREND_SCORE_ENTRY) &
        (df["trend_score"] < df["trend_score"].shift(1))
    )
    return df


# ══════════════════════════════════════════════════════════════════════════════
# TRADE DOCUMENT HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_document(symbol, entry_price, quantity, trend_score_at_entry,
                          entry_score=0, entry_details=None):
    return {
        "instrument":        symbol,
        "direction":         "long",
        "entry_price":       entry_price,
        "highest_price":     entry_price,
        "quantity":          quantity,
        "entry_timestamp":   datetime.now(),
        "entry_trend_score": trend_score_at_entry,
        "entry_score":       entry_score,
        "entry_details":     entry_details or {},
        "order_id":          str(ObjectId()),
        "exit_price":        None,
        "exit_timestamp":    None,
        "reason":            None,
        "current_pnl":       0,
        "realized_pnl":      0,
        "status":            "open",
        "phase":             "initial",
        "created_at":        datetime.now(),
        "updated_at":        datetime.now(),
    }


def update_trade_document(trade_id, updates: dict):
    updates["updated_at"] = datetime.now()
    trades_col.update_one({"_id": ObjectId(trade_id)}, {"$set": updates})


# ══════════════════════════════════════════════════════════════════════════════
# TRADING LOOP  (runs as asyncio task every 5 minutes)
# ══════════════════════════════════════════════════════════════════════════════

async def trading_loop():
    log.info("Trading loop started.")
    await asyncio.sleep(10)   # brief startup grace period

    while True:
        # ── EOD cleanup ───────────────────────────────────────────────────────
        watchlist_clear_eod()
        watchlist_print()

        # Log current session type
        rth = is_rth()
        et_now = _get_eastern_now()
        log.info(f"Session: {'RTH' if rth else 'EXTENDED HOURS'} "
                 f"(ET: {et_now.strftime('%H:%M')})")

        # ── Load tickers dynamically from MongoDB ─────────────────────────────
        symbols = watchlist_get_all()

        if not symbols:
            log.info("Watchlist empty — nothing to evaluate. Waiting 5 min...")
            await asyncio.sleep(300)
            continue

        log.info(f"Evaluating {len(symbols)} ticker(s): {', '.join(symbols)}")

        contracts = [Stock(s, "SMART", "USD") for s in symbols]
        try:
            ib.qualifyContracts(*contracts)
        except Exception as e:
            log.warning(f"qualifyContracts error: {e}")

        for contract in contracts:
            symbol = contract.symbol
            log.info(f"\n{'='*60}")
            log.info(f"Evaluating {symbol} at {datetime.now().strftime('%H:%M:%S')}")
            log.info(f"{'='*60}")

            # ── Fetch 5-min bars ──────────────────────────────────────────────
            try:
                bars = ib.reqHistoricalData(
                    contract,
                    endDateTime="",
                    durationStr="2 D",
                    barSizeSetting="5 mins",
                    whatToShow="TRADES",
                    useRTH=False,
                    formatDate=1,
                )
            except Exception as e:
                log.warning(f"{symbol}: reqHistoricalData error — {e}")
                continue

            if not bars:
                log.warning(f"{symbol}: no historical data returned")
                continue

            df = util.df(bars)
            df.columns = [c.lower() for c in df.columns]

            # ── Calculate ALL indicators ──────────────────────────────────────
            df = calculate_ema_alignment_oscillator(df)
            df = calculate_ema_200(df)
            df = calculate_session_vwap(df)
            df = calculate_psar(df)
            df["vol_heatmap"] = volume_heatmap(df)
            df = calculate_ema_9_21(df)

            row           = df.iloc[-1]
            current_price = row["close"]
            trend_score   = row["trend_score"]
            atr           = row.get("ATR", BASE_TRAILING_AMT)
            vwap          = row["vwap"]

            entry_score, entry_details = compute_entry_score(row)

            log.info(f"Price: ${current_price:.2f} | TrendScore: {trend_score:.1f} | "
                     f"EMA200: ${row['ema_200']:.2f} | VWAP: ${vwap:.2f}")
            log.info(f"EntryScore: {entry_score}/3 | "
                     f"PSAR={'✓' if entry_details['psar_bullish'] else '✗'} | "
                     f"VolHeat={'✓' if entry_details['extra_high_vol_up'] else '✗'} ({row['vol_heatmap']}) | "
                     f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '✗'}")

            open_trade = trades_col.find_one({"instrument": symbol, "status": "open"})

            # ═════════════════════════════════════════════════════════════════
            # SELL / POSITION MANAGEMENT
            # ═════════════════════════════════════════════════════════════════

            if open_trade:
                entry_price   = open_trade["entry_price"]
                highest_price = open_trade.get("highest_price", entry_price)
                quantity      = open_trade["quantity"]
                time_in_trade = (datetime.now() - open_trade["entry_timestamp"]).total_seconds() / 3600

                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade["_id"], {"highest_price": highest_price})

                profit_pct  = (current_price - entry_price) / entry_price
                pnl_ps      = current_price - entry_price
                current_pnl = pnl_ps * quantity
                update_trade_document(open_trade["_id"], {"current_pnl": current_pnl})

                log.info(f"OPEN | Entry: ${entry_price:.2f} | P&L: {profit_pct:.2%} | Score: {trend_score:.1f}")

                # ── Stop loss ─────────────────────────────────────────────────
                should_exit = False
                exit_reason = None

                if profit_pct <= -STOP_LOSS_1:
                    if trend_score < TREND_SCORE_DROP_EXIT:
                        should_exit = True
                        exit_reason = "stop_loss_10pct_trend_weak"
                        log.info(f"EXIT: 10% loss + score {trend_score:.1f} < {TREND_SCORE_DROP_EXIT}")
                    elif profit_pct <= -STOP_LOSS_2:
                        should_exit = True
                        exit_reason = "stop_loss_15pct_hard"
                        log.info(f"EXIT: 15% hard stop reached")
                    else:
                        log.info(f"Holding at {profit_pct:.2%} loss — trend still strong ({trend_score:.1f})")

                if should_exit and quantity > 0:
                    sell_price = get_bid_price(contract) or current_price
                    order = create_sell_order(quantity, limit_price=sell_price)
                    ib.placeOrder(contract, order)

                    realized = pnl_ps * quantity
                    update_trade_document(open_trade["_id"], {
                        "exit_price":       current_price,
                        "exit_timestamp":   datetime.now(),
                        "reason":           exit_reason,
                        "realized_pnl":     realized,
                        "exit_trend_score": trend_score,
                        "status":           "closed",
                    })
                    log.info(f"STOP LOSS | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                    await asyncio.sleep(0.5)
                    continue

                # ── Take profit ───────────────────────────────────────────────
                if profit_pct >= TAKE_PROFIT_PCT:
                    sell_price = get_bid_price(contract) or current_price
                    order = create_sell_order(quantity, limit_price=sell_price)
                    ib.placeOrder(contract, order)

                    realized = pnl_ps * quantity
                    update_trade_document(open_trade["_id"], {
                        "exit_price":       current_price,
                        "exit_timestamp":   datetime.now(),
                        "reason":           f"take_profit_{profit_pct:.2%}",
                        "realized_pnl":     realized,
                        "exit_trend_score": trend_score,
                        "status":           "closed",
                    })
                    log.info(f"TAKE PROFIT | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                    await asyncio.sleep(0.5)
                    continue

                # ── Trailing stop (only when profitable) ──────────────────────
                if profit_pct > 0:
                    atr_stop  = highest_price - (atr * ATR_MULTIPLIER)
                    pct_stop  = highest_price * (1 - MIN_TRAILING_PCT)
                    fix_stop  = highest_price - BASE_TRAILING_AMT
                    t_factor  = min(1.0, time_in_trade / 24)
                    trail_stop = max(atr_stop * (1 - 0.25 * t_factor), pct_stop, fix_stop)

                    if current_price <= trail_stop and quantity > 0:
                        sell_price = get_bid_price(contract) or current_price
                        order = create_sell_order(quantity, limit_price=sell_price)
                        ib.placeOrder(contract, order)

                        realized = pnl_ps * quantity
                        update_trade_document(open_trade["_id"], {
                            "exit_price":       current_price,
                            "exit_timestamp":   datetime.now(),
                            "reason":           "trailing_stop_hit",
                            "realized_pnl":     realized,
                            "exit_trend_score": trend_score,
                            "status":           "closed",
                        })
                        log.info(f"TRAILING STOP | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                        await asyncio.sleep(0.5)
                        continue

            # ═════════════════════════════════════════════════════════════════
            # BUY / ENTRY LOGIC  (v2 — scoring-based)
            # ═════════════════════════════════════════════════════════════════

            else:
                if entry_score < ENTRY_SCORE_MIN:
                    log.info(
                        f"{symbol}: entry score {entry_score}/3 < {ENTRY_SCORE_MIN} — "
                        f"PSAR={'✓' if entry_details['psar_bullish'] else '✗'} | "
                        f"VolHeat={'✓' if entry_details['extra_high_vol_up'] else '✗'} | "
                        f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '✗'} "
                        f"— will re-check next cycle"
                    )
                    await asyncio.sleep(0.5)
                    continue

                log.info(
                    f"{symbol}: ENTRY SCORE {entry_score}/3 ≥ {ENTRY_SCORE_MIN} ✓  "
                    f"PSAR={'✓' if entry_details['psar_bullish'] else '✗'} | "
                    f"VolHeat={'✓' if entry_details['extra_high_vol_up'] else '✗'} | "
                    f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '✗'}"
                )

                if has_open_position(symbol):
                    log.info(f"{symbol}: Gate 1 FAILED — existing IB position — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue

                if has_pending_order(symbol):
                    log.info(f"{symbol}: Gate 2 FAILED — pending IB order — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue

                ask_price = get_ask_price(contract)
                if not ask_price:
                    log.info(f"{symbol}: Gate 4 FAILED — no valid ask price — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue

                # ── Place order (RTH-aware) ───────────────────────────────────
                order = create_buy_order(ORDER_SIZE, limit_price=ask_price)
                ib.placeOrder(contract, order)

                trade_doc = create_trade_document(
                    symbol               = symbol,
                    entry_price          = ask_price,
                    quantity             = ORDER_SIZE,
                    trend_score_at_entry = trend_score,
                    entry_score          = entry_score,
                    entry_details        = entry_details,
                )
                trades_col.insert_one(trade_doc)

                log.info(
                    f"BUY | {symbol} | ask=${ask_price:.2f} | qty={ORDER_SIZE} | "
                    f"session={'RTH' if is_rth() else 'EXT'} | "
                    f"score={entry_score}/3 | trend={trend_score:.1f} | "
                    f"PSAR ✓ | VolHeat={'✓' if entry_details['extra_high_vol_up'] else '—'} | "
                    f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '—'} | "
                    f"G1 ✓ | G2 ✓ | G4 ✓"
                )

            await asyncio.sleep(0.5)

        # ── End of scan cycle ─────────────────────────────────────────────────
        log.info(f"\n{'='*60}")
        log.info("Scan complete. Next scan in 5 minutes.")
        watchlist_print()
        log.info(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# TELEGRAM POLLER
# ══════════════════════════════════════════════════════════════════════════════

def load_session() -> str:
    if os.path.exists(SESSION_FILE):
        with open(SESSION_FILE, encoding="utf-8") as f:
            return f.read().strip()
    return ""


def save_session(s: str):
    with open(SESSION_FILE, "w", encoding="utf-8") as f:
        f.write(s)
    log.info(f"Telegram session saved → {SESSION_FILE}")


async def telegram_poller():
    log.info("Telegram poller started.")

    while True:
        tg = TelegramClient(StringSession(load_session()), API_ID, API_HASH)
        try:
            await tg.start(phone=PHONE)
            save_session(tg.session.save())

            m = re.search(r't\.me/(?:joinchat/|\+)([A-Za-z0-9_-]+)', INVITE_LINK)
            if not m:
                raise ValueError(f"Cannot parse invite hash: {INVITE_LINK}")
            try:
                await tg(ImportChatInviteRequest(m.group(1)))
                log.info("Joined Telegram channel")
            except UserAlreadyParticipantError:
                log.info("Already a member of the channel")

            channel      = await tg.get_entity(INVITE_LINK)
            last_seen_id = None
            log.info(f"Polling channel: {channel.title} (ID: {channel.id})")

            while True:
                messages = await tg.get_messages(channel, limit=1)
                if messages:
                    msg    = messages[0]
                    is_new = (last_seen_id is None or msg.id != last_seen_id)

                    if msg.text and is_new:
                        parsed = parse_structured_message(msg.text)
                        if parsed:
                            symbol = parsed["symbol"]
                            status = parsed["status"]
                        else:
                            symbol = extract_ticker(msg.text)
                            status = "NEW"

                        preview = msg.text[:120] + ("..." if len(msg.text) > 120 else "")
                        safe_print(f"\n[{msg.date}]")
                        safe_print(f"   Message : {preview}")
                        safe_print(f"   Ticker  : {symbol or '(not found)'}")
                        safe_print(f"   Status  : NEW")

                        if symbol and is_new_signal(status):
                            inserted = watchlist_add(symbol, source="telegram")
                            if inserted:
                                safe_print(f"   → Added {symbol} to watchlist_tickers in MongoDB")
                            else:
                                safe_print(f"   → {symbol} already in watchlist_tickers, skipped")
                        elif symbol:
                            log.info(f"{symbol}: status='{status}' — not NEW, skipping watchlist add")

                        last_seen_id = msg.id

                log.debug(f"Telegram: waiting {POLL_INTERVAL}s...")
                await asyncio.sleep(POLL_INTERVAL)

        except FloodWaitError as e:
            log.warning(f"Telegram rate-limited — waiting {e.seconds}s")
            await asyncio.sleep(e.seconds + 5)
        except KeyboardInterrupt:
            log.info("Telegram poller stopped by user.")
            break
        except Exception as e:
            log.error(f"Telegram error: {type(e).__name__}: {e} — reconnecting in 15s")
            await asyncio.sleep(15)
        finally:
            try:
                await tg.disconnect()
            except Exception:
                pass


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

async def main():
    log.info("=== Unified Trading Bot Starting ===")
    ibkr_connect()
    await asyncio.gather(
        telegram_poller(),
        trading_loop(),
    )


if __name__ == "__main__":
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        log.info("Bot stopped by user.")
    finally:
        ib.disconnect()
        log.info("IBKR disconnected.")


In [ ]:
"""
Unified Trading Bot v2.1
════════════════════════
Telegram Poller  ──►  MongoDB (watchlist_tickers)  ──►  IBKR Trading Loop

CHANGES from v2.0:
──────────────────
FIX 1: DUPLICATE ENTRY PREVENTION
  - Added Gate 0: checks MongoDB trades_v2 for any open trade on the
    same symbol BEFORE checking IBKR state. This is the authoritative
    gate — no more race conditions from IBKR fill latency.
  - Gate 1 (IBKR position check) kept as secondary safety net.

FIX 2: RE-ENTRY COOLDOWN CHECK
  - Before entering, checks the cooldown collection set by the sell bot.
    If the sell bot recently closed a position on this symbol, the buy
    bot waits until cooldown expires.

FIX 3: POSITION SIZING BY DOLLAR AMOUNT
  - ORDER_SIZE is now a dollar budget ($500), not a share count.
    qty = int(ORDER_BUDGET / ask_price)
    This prevents tiny $10 positions on sub-$1 stocks and oversized
    positions on expensive stocks.

Flow:
  1. Telethon polls a private Telegram channel every 20 seconds.
  2. NEW signals are parsed for a ticker symbol and upserted into
     the 'watchlist_tickers' MongoDB collection.
  3. The IBKR trading loop runs every 5 minutes, reads ALL active
     tickers from 'watchlist_tickers', qualifies contracts on-the-fly,
     and evaluates buy / position-management logic for each.
  4. At EOD (>= 4 PM) the watchlist_tickers collection is cleared.

ENTRY SCORING (v2):
  Each ticker is scored on 3 conditions (1 point each):
    1. Parabolic SAR indicates bullish entry (SAR below price)
    2. Volume Heatmap shows "Extra High Volume Up"
    3. Close > EMA 9 > EMA 21  (bullish EMA stack)
  Score >= 2  →  enter position

ORDER ROUTING:
  - Regular Trading Hours (9:30–16:00 ET): MarketOrder
  - Premarket / After-hours: LimitOrder at ask price + 1% slippage,
    outsideRth=True, tif=DAY
"""

import asyncio
import os
import re
import sys
import random
import logging
import warnings
from datetime import datetime, timezone, timedelta

import nest_asyncio
nest_asyncio.apply()

import numpy as np
import pandas as pd
from pymongo import MongoClient
from bson import ObjectId

from ib_insync import IB, Stock, MarketOrder, LimitOrder, util

from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import FloodWaitError, UserAlreadyParticipantError
from telethon.tl.functions.messages import ImportChatInviteRequest

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt    = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh     = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh     = logging.FileHandler("unified_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("unified_bot")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


def safe_print(text: str):
    try:
        encoded = text.encode(sys.stdout.encoding or "utf-8", errors="replace")
        sys.stdout.buffer.write(encoded + b"\n")
        sys.stdout.buffer.flush()
    except Exception:
        print(text)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

# Telegram
API_ID        = 34025130
API_HASH      = "e5e514ebcb8a15042feadfecd7a15cfe"
PHONE         = "+18477675748"
INVITE_LINK   = "https://t.me/+M5HRCxZycqJmOTg5"
POLL_INTERVAL = 20
SESSION_FILE  = "unified_bot_session.txt"

# IBKR
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)

# ─── Position Sizing (v2.1 — dollar-based instead of fixed share count) ──────
ORDER_BUDGET   = 500          # $500 per position
MIN_ORDER_QTY  = 5            # never buy fewer than 5 shares
MAX_ORDER_QTY  = 10000        # cap for very cheap stocks

# Trading parameters
TREND_SCORE_ENTRY    = 85
TREND_SCORE_WATCHLIST = 85
TREND_SCORE_DROP_EXIT = 69
STOP_LOSS_1          = 0.10
STOP_LOSS_2          = 0.15
TAKE_PROFIT_PCT      = 0.50
BASE_TRAILING_AMT    = 1.0
MIN_TRAILING_PCT     = 0.005
ATR_MULTIPLIER       = 1.5
EOD_CLEAR_HOUR       = 16

# Entry scoring
ENTRY_SCORE_MIN      = 2

# PSAR parameters
PSAR_AF_START        = 0.02
PSAR_AF_INCREMENT    = 0.02
PSAR_AF_MAX          = 0.20

# RTH hours (Eastern Time)
RTH_START_HOUR       = 9
RTH_START_MINUTE     = 30
RTH_END_HOUR         = 16
RTH_END_MINUTE       = 0

# Limit order settings for extended hours
LIMIT_PRICE_SLIPPAGE = 0.01


# ══════════════════════════════════════════════════════════════════════════════
# ORDER ROUTING — RTH vs Extended Hours
# ══════════════════════════════════════════════════════════════════════════════

def _get_eastern_now() -> datetime:
    utc_now = datetime.now(timezone.utc)
    month = utc_now.month
    if 3 <= month <= 10:
        offset = timedelta(hours=-4)   # EDT
    else:
        offset = timedelta(hours=-5)   # EST
    return utc_now + offset


def is_rth() -> bool:
    et_now = _get_eastern_now()
    rth_open  = et_now.replace(hour=RTH_START_HOUR, minute=RTH_START_MINUTE, second=0, microsecond=0)
    rth_close = et_now.replace(hour=RTH_END_HOUR,   minute=RTH_END_MINUTE,   second=0, microsecond=0)
    return rth_open <= et_now < rth_close


def create_buy_order(qty: int, limit_price: float = None):
    if is_rth():
        log.info(f"ORDER: MarketOrder BUY x{qty} (RTH)")
        return MarketOrder("BUY", qty)
    else:
        if limit_price is None:
            raise ValueError("limit_price required for extended-hours BUY order")
        aggressive_price = round(limit_price * (1 + LIMIT_PRICE_SLIPPAGE), 2)
        order = LimitOrder("BUY", qty, aggressive_price)
        order.outsideRth = True
        order.tif = "DAY"
        log.info(f"ORDER: LimitOrder BUY x{qty} @ ${aggressive_price:.2f} "
                 f"(extended hours, ask=${limit_price:.2f} + {LIMIT_PRICE_SLIPPAGE*100:.0f}% slip)")
        return order


def create_sell_order(qty: int, limit_price: float = None):
    if is_rth():
        log.info(f"ORDER: MarketOrder SELL x{qty} (RTH)")
        return MarketOrder("SELL", qty)
    else:
        if limit_price is None:
            raise ValueError("limit_price required for extended-hours SELL order")
        aggressive_price = round(limit_price * (1 - LIMIT_PRICE_SLIPPAGE), 2)
        order = LimitOrder("SELL", qty, aggressive_price)
        order.outsideRth = True
        order.tif = "DAY"
        log.info(f"ORDER: LimitOrder SELL x{qty} @ ${aggressive_price:.2f} "
                 f"(extended hours, price=${limit_price:.2f} - {LIMIT_PRICE_SLIPPAGE*100:.0f}% slip)")
        return order


# ══════════════════════════════════════════════════════════════════════════════
# POSITION SIZING (v2.1)
# ══════════════════════════════════════════════════════════════════════════════

def calculate_order_qty(price: float) -> int:
    """
    Compute share count from dollar budget.
    $500 budget at $0.50/share = 1000 shares (one real position)
    $500 budget at $25/share   = 20 shares
    """
    if price <= 0:
        return 0
    qty = int(ORDER_BUDGET / price)
    qty = max(qty, MIN_ORDER_QTY)
    qty = min(qty, MAX_ORDER_QTY)
    return qty


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB
# ══════════════════════════════════════════════════════════════════════════════

mongo_client = MongoClient("mongodb://localhost:27017/")
db           = mongo_client["breakout_db"]

watchlist_col   = db["watchlist_tickers"]
trades_col      = db["trades_v2"]
excluded_col    = db["excluded_tickers_v2"]
cooldown_col    = db["cooldowns"]           # shared with sell bot

watchlist_col.create_index("symbol", unique=True)


# ── Watchlist helpers ─────────────────────────────────────────────────────────

def watchlist_add(symbol: str, source: str = "telegram") -> bool:
    try:
        watchlist_col.insert_one({
            "symbol":     symbol,
            "source":     source,
            "added_at":   datetime.now(timezone.utc),
            "active":     True,
        })
        log.info(f"WATCHLIST ADD  : {symbol}  (source={source})")
        return True
    except Exception:
        log.debug(f"WATCHLIST DUP  : {symbol} already present, skipped.")
        return False


def watchlist_get_all() -> list[str]:
    docs = watchlist_col.find({"active": True}, {"symbol": 1})
    return [d["symbol"] for d in docs]


def watchlist_clear_eod():
    now = datetime.now()
    if now.hour >= EOD_CLEAR_HOUR:
        result = watchlist_col.delete_many({})
        if result.deleted_count:
            log.info(f"EOD CLEAR: removed {result.deleted_count} ticker(s) from watchlist_tickers")


def watchlist_print():
    symbols = watchlist_get_all()
    if symbols:
        log.info(f"Watchlist ({len(symbols)}): {', '.join(symbols)}")
    else:
        log.info("Watchlist: empty")


# ── Duplicate & Cooldown Checks (NEW in v2.1) ────────────────────────────────

def has_open_trade_in_db(symbol: str) -> bool:
    """
    Gate 0 — THE authoritative duplicate check.
    Queries MongoDB for any open or merged trade on this symbol.
    This prevents the race condition where IBKR hasn't filled yet
    but we already inserted a trade document.
    """
    existing = trades_col.find_one({
        "instrument": symbol,
        "status": {"$in": ["open", "merged"]},
    })
    if existing:
        log.info(f"{symbol}: Gate 0 BLOCKED — already have open trade in DB "
                 f"(id={existing['_id']}, qty={existing.get('quantity', '?')})")
        return True
    return False


def is_in_cooldown(symbol: str) -> bool:
    """
    Check sell bot's cooldown collection.
    If the sell bot recently closed this symbol, don't re-enter yet.
    """
    record = cooldown_col.find_one({"symbol": symbol})
    if not record:
        return False

    expires_at = record.get("expires_at")
    if not expires_at:
        return False

    now = datetime.now(timezone.utc)
    if expires_at.tzinfo is None:
        in_cooldown = datetime.now() < expires_at
    else:
        in_cooldown = now < expires_at

    if in_cooldown:
        remaining = (expires_at - now).total_seconds() / 60
        log.info(f"{symbol}: COOLDOWN ACTIVE — {remaining:.1f} min remaining "
                 f"(closed for: {record.get('reason', 'unknown')})")
        return True
    else:
        # Expired — clean up
        cooldown_col.delete_one({"symbol": symbol})
        return False


# ══════════════════════════════════════════════════════════════════════════════
# TELEGRAM PARSERS
# ══════════════════════════════════════════════════════════════════════════════

def extract_ticker(text: str) -> str | None:
    text = text.strip()
    m = re.match(r'^\[.*?\]\s+([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    m = re.match(r'^([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    return None


def parse_structured_message(text: str) -> dict | None:
    result = {"symbol": None, "status": None}
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"(?i)^ticker\s*:", line):
            result["symbol"] = line.split(":", 1)[1].strip().upper()
        if re.match(r"(?i)^status\s*:", line):
            result["status"] = line.split(":", 1)[1].strip()
    if result["symbol"] and result["status"]:
        return result
    return None


def is_new_signal(status_text: str) -> bool:
    return "new" in status_text.lower()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

ib = IB()

def ibkr_connect():
    util.startLoop()
    ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
    log.info(f"IBKR connected | accounts: {ib.wrapper.accounts}")


# ── Gate helpers ──────────────────────────────────────────────────────────────

def has_open_position(symbol: str) -> bool:
    result = any(p.contract.symbol == symbol and p.position != 0 for p in ib.positions())
    log.debug(f"Gate 1 | has_open_position({symbol}) => {result}")
    return result


def has_pending_order(symbol: str) -> bool:
    result = any(t.contract.symbol == symbol for t in ib.openTrades())
    log.debug(f"Gate 2 | has_pending_order({symbol}) => {result}")
    return result


def get_ask_price(contract) -> float | None:
    ticker = ib.reqMktData(contract, "106", False, False)
    ib.sleep(3)
    ib.cancelMktData(contract)
    for label, val in [("ask", ticker.ask), ("last", ticker.last), ("close", ticker.close)]:
        if val and float(val) > 0:
            log.info(f"Gate 4 | {contract.symbol}: source='{label}'  ${float(val):.4f}")
            return float(val)
    log.warning(f"Gate 4 | {contract.symbol}: all price fields empty")
    return None


def get_bid_price(contract) -> float | None:
    ticker = ib.reqMktData(contract, "106", False, False)
    ib.sleep(3)
    ib.cancelMktData(contract)
    for label, val in [("bid", ticker.bid), ("last", ticker.last), ("close", ticker.close)]:
        if val and float(val) > 0:
            log.info(f"Bid | {contract.symbol}: source='{label}'  ${float(val):.4f}")
            return float(val)
    log.warning(f"Bid | {contract.symbol}: all price fields empty")
    return None


# ══════════════════════════════════════════════════════════════════════════════
# INDICATOR CALCULATIONS
# ══════════════════════════════════════════════════════════════════════════════

def calculate_ema(df: pd.DataFrame, period: int) -> pd.Series:
    return df["close"].ewm(span=period, adjust=False).mean()


def calculate_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    high_low   = df["high"] - df["low"]
    high_close = np.abs(df["high"] - df["close"].shift())
    low_close  = np.abs(df["low"]  - df["close"].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["ATR"]  = true_range.rolling(period).mean()
    return df


def calculate_ema_200(df: pd.DataFrame) -> pd.DataFrame:
    df["ema_200"] = calculate_ema(df, 200)
    return df


def calculate_session_vwap(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date_only"] = pd.to_datetime(df["date"]).dt.date
    today           = df["date_only"].max()
    mask            = df["date_only"] == today
    today_df        = df.loc[mask].copy()
    today_df["vwap"] = (
        (today_df["close"] * today_df["volume"]).cumsum() /
        today_df["volume"].cumsum()
    )
    df.loc[mask, "vwap"] = today_df["vwap"].values
    df["vwap"] = df["vwap"].ffill()
    return df


# ── Parabolic SAR ────────────────────────────────────────────────────────────

def calculate_psar(df: pd.DataFrame,
                   af_start: float = PSAR_AF_START,
                   af_increment: float = PSAR_AF_INCREMENT,
                   af_max: float = PSAR_AF_MAX) -> pd.DataFrame:
    high   = df["high"].values
    low    = df["low"].values
    close  = df["close"].values
    n      = len(df)

    psar          = np.zeros(n, dtype=float)
    psar_bullish  = np.zeros(n, dtype=bool)
    af            = af_start
    ep            = 0.0
    bull          = True

    psar[0]         = low[0]
    psar_bullish[0] = True
    ep              = high[0]

    for i in range(1, n):
        psar[i] = psar[i - 1] + af * (ep - psar[i - 1])

        if bull:
            psar[i] = min(psar[i], low[i - 1])
            if i >= 2:
                psar[i] = min(psar[i], low[i - 2])
            if low[i] < psar[i]:
                bull    = False
                psar[i] = ep
                ep      = low[i]
                af      = af_start
            else:
                if high[i] > ep:
                    ep = high[i]
                    af = min(af + af_increment, af_max)
        else:
            psar[i] = max(psar[i], high[i - 1])
            if i >= 2:
                psar[i] = max(psar[i], high[i - 2])
            if high[i] > psar[i]:
                bull    = True
                psar[i] = ep
                ep      = high[i]
                af      = af_start
            else:
                if low[i] < ep:
                    ep = low[i]
                    af = min(af + af_increment, af_max)

        psar_bullish[i] = bull

    df["psar"]         = psar
    df["psar_bullish"] = psar_bullish
    return df


# ── Volume Heatmap ───────────────────────────────────────────────────────────

def volume_heatmap(df: pd.DataFrame,
                   length: int = 60,
                   slength: int = 60,
                   threshold_extra_high: float = 4.0,
                   threshold_high: float = 2.5,
                   threshold_medium: float = 1.0,
                   threshold_normal: float = -0.5) -> pd.Series:
    mean   = df["volume"].rolling(window=length).mean()
    std    = df["volume"].rolling(window=slength).std()
    stdbar = (df["volume"] - mean) / std
    direction = df["close"] > df["open"]

    def classify(row):
        if pd.isna(row["stdbar"]):
            return "Insufficient Data"
        tag = " Up" if row["direction"] else " Down"
        if row["stdbar"] > threshold_extra_high:
            return "Extra High Volume" + tag
        elif row["stdbar"] > threshold_high:
            return "High Volume" + tag
        elif row["stdbar"] > threshold_medium:
            return "Medium Volume" + tag
        elif row["stdbar"] > threshold_normal:
            return "Normal Volume" + tag
        else:
            return "Low Volume" + tag

    temp = pd.DataFrame({"stdbar": stdbar, "direction": direction})
    return temp.apply(classify, axis=1)


# ── EMA 9 / 21 stack check ──────────────────────────────────────────────────

def calculate_ema_9_21(df: pd.DataFrame) -> pd.DataFrame:
    df["ema_9"]  = df["close"].ewm(span=9,  adjust=False).mean()
    df["ema_21"] = df["close"].ewm(span=21, adjust=False).mean()
    df["ema_9_21_bullish"] = (df["close"] > df["ema_9"]) & (df["ema_9"] > df["ema_21"])
    return df


# ── Combined entry score ─────────────────────────────────────────────────────

def compute_entry_score(row: pd.Series) -> tuple[int, dict]:
    cond_psar   = bool(row.get("psar_bullish", False))
    cond_volume = str(row.get("vol_heatmap", "")).startswith("Extra High Volume Up")
    cond_ema    = bool(row.get("ema_9_21_bullish", False))

    details = {
        "psar_bullish":        cond_psar,
        "extra_high_vol_up":   cond_volume,
        "ema_9_21_stack":      cond_ema,
    }
    score = int(cond_psar) + int(cond_volume) + int(cond_ema)
    return score, details


def calculate_ema_alignment_oscillator(df: pd.DataFrame,
                                       fast_period=8,
                                       mid_period=13,
                                       slow_period=14,
                                       trend_period=21,
                                       slope_threshold=1.0,
                                       spacing_min=0.2) -> pd.DataFrame:
    df["ema_fast"]  = calculate_ema(df, fast_period)
    df["ema_mid"]   = calculate_ema(df, mid_period)
    df["ema_slow"]  = calculate_ema(df, slow_period)
    df["ema_trend"] = calculate_ema(df, trend_period)
    df = calculate_atr(df, 14)

    def slope_deg(ema_s, atr_s):
        s = (ema_s - ema_s.shift(1)) / (atr_s + 0.0001) * 100
        return np.degrees(np.arctan(s))

    df["fast_angle"] = slope_deg(df["ema_fast"], df["ATR"])
    df["mid_angle"]  = slope_deg(df["ema_mid"],  df["ATR"])
    df["slow_angle"] = slope_deg(df["ema_slow"], df["ATR"])

    df["bullish_stack"] = (
        (df["ema_fast"]  > df["ema_mid"]) &
        (df["ema_mid"]   > df["ema_slow"]) &
        (df["ema_slow"]  > df["ema_trend"])
    )
    df["bearish_stack"] = (
        (df["ema_fast"]  < df["ema_mid"]) &
        (df["ema_mid"]   < df["ema_slow"]) &
        (df["ema_slow"]  < df["ema_trend"])
    )

    df["price_above_fast"] = df["close"] > df["ema_fast"]
    df["price_below_fast"] = df["close"] < df["ema_fast"]

    df["spacing_fast_mid"] = np.abs(df["ema_fast"] - df["ema_mid"]) / df["ema_mid"] * 100
    df["spacing_mid_slow"] = np.abs(df["ema_mid"]  - df["ema_slow"]) / df["ema_slow"] * 100
    df["avg_spacing"]      = (df["spacing_fast_mid"] + df["spacing_mid_slow"]) / 2

    df["volume_sma"]          = df["volume"].rolling(20).mean()
    df["volume_confirmation"] = df["volume"] > df["volume_sma"]

    df["trend_score"] = 0.0

    for i in range(1, len(df)):
        prev = df["trend_score"].iloc[i - 1]

        if df["bullish_stack"].iloc[i] and df["price_above_fast"].iloc[i]:
            avg_slope     = (df["fast_angle"].iloc[i] + df["mid_angle"].iloc[i] + df["slow_angle"].iloc[i]) / 3
            slope_bonus   = min(avg_slope * 3, 30)  if avg_slope > slope_threshold else 0
            spacing_bonus = min(df["avg_spacing"].iloc[i] * 10, 20) if df["avg_spacing"].iloc[i] > spacing_min else 0
            vol_bonus     = 10 if df["volume_confirmation"].iloc[i] else 0
            new_score     = 40.0 + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], "trend_score"] = (new_score + prev) / 2

        elif df["bearish_stack"].iloc[i] and df["price_below_fast"].iloc[i]:
            avg_slope     = (abs(df["fast_angle"].iloc[i]) + abs(df["mid_angle"].iloc[i]) + abs(df["slow_angle"].iloc[i])) / 3
            slope_bonus   = -min(avg_slope * 3, 30)  if avg_slope > slope_threshold else 0
            spacing_bonus = -min(df["avg_spacing"].iloc[i] * 10, 20) if df["avg_spacing"].iloc[i] > spacing_min else 0
            vol_bonus     = -10 if df["volume_confirmation"].iloc[i] else 0
            new_score     = -40.0 + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], "trend_score"] = (new_score + prev) / 2

        else:
            df.loc[df.index[i], "trend_score"] = prev * 0.9

    df.loc[~df["volume_confirmation"], "trend_score"] *= 0.8
    df["trend_score"] = df["trend_score"].clip(-100, 100)

    df["strong_bull_signal"] = (
        (df["trend_score"] >= TREND_SCORE_ENTRY) &
        (df["trend_score"] > df["trend_score"].shift(1))
    )
    df["strong_bear_signal"] = (
        (df["trend_score"] <= -TREND_SCORE_ENTRY) &
        (df["trend_score"] < df["trend_score"].shift(1))
    )
    return df


# ══════════════════════════════════════════════════════════════════════════════
# TRADE DOCUMENT HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_document(symbol, entry_price, quantity, trend_score_at_entry,
                          entry_score=0, entry_details=None):
    return {
        "instrument":        symbol,
        "direction":         "long",
        "entry_price":       entry_price,
        "highest_price":     entry_price,
        "quantity":          quantity,
        "entry_timestamp":   datetime.now(),
        "entry_trend_score": trend_score_at_entry,
        "entry_score":       entry_score,
        "entry_details":     entry_details or {},
        "order_id":          str(ObjectId()),
        "exit_price":        None,
        "exit_timestamp":    None,
        "reason":            None,
        "current_pnl":       0,
        "realized_pnl":      0,
        "status":            "open",
        "phase":             "initial",
        "created_at":        datetime.now(),
        "updated_at":        datetime.now(),
    }


def update_trade_document(trade_id, updates: dict):
    updates["updated_at"] = datetime.now()
    trades_col.update_one({"_id": ObjectId(trade_id)}, {"$set": updates})


# ══════════════════════════════════════════════════════════════════════════════
# TRADING LOOP  (runs as asyncio task every 5 minutes)
# ══════════════════════════════════════════════════════════════════════════════

async def trading_loop():
    log.info("Trading loop started.")
    await asyncio.sleep(10)

    while True:
        watchlist_clear_eod()
        watchlist_print()

        rth = is_rth()
        et_now = _get_eastern_now()
        log.info(f"Session: {'RTH' if rth else 'EXTENDED HOURS'} "
                 f"(ET: {et_now.strftime('%H:%M')})")

        symbols = watchlist_get_all()

        if not symbols:
            log.info("Watchlist empty — nothing to evaluate. Waiting 5 min...")
            await asyncio.sleep(300)
            continue

        log.info(f"Evaluating {len(symbols)} ticker(s): {', '.join(symbols)}")

        contracts = [Stock(s, "SMART", "USD") for s in symbols]
        try:
            ib.qualifyContracts(*contracts)
        except Exception as e:
            log.warning(f"qualifyContracts error: {e}")

        for contract in contracts:
            symbol = contract.symbol
            log.info(f"\n{'='*60}")
            log.info(f"Evaluating {symbol} at {datetime.now().strftime('%H:%M:%S')}")
            log.info(f"{'='*60}")

            # ── Fetch 5-min bars ──────────────────────────────────────────────
            try:
                bars = ib.reqHistoricalData(
                    contract,
                    endDateTime="",
                    durationStr="2 D",
                    barSizeSetting="5 mins",
                    whatToShow="TRADES",
                    useRTH=False,
                    formatDate=1,
                )
            except Exception as e:
                log.warning(f"{symbol}: reqHistoricalData error — {e}")
                continue

            if not bars:
                log.warning(f"{symbol}: no historical data returned")
                continue

            df = util.df(bars)
            df.columns = [c.lower() for c in df.columns]

            # ── Calculate ALL indicators ──────────────────────────────────────
            df = calculate_ema_alignment_oscillator(df)
            df = calculate_ema_200(df)
            df = calculate_session_vwap(df)
            df = calculate_psar(df)
            df["vol_heatmap"] = volume_heatmap(df)
            df = calculate_ema_9_21(df)

            row           = df.iloc[-1]
            current_price = row["close"]
            trend_score   = row["trend_score"]
            atr           = row.get("ATR", BASE_TRAILING_AMT)
            vwap          = row["vwap"]

            entry_score, entry_details = compute_entry_score(row)

            log.info(f"Price: ${current_price:.2f} | TrendScore: {trend_score:.1f} | "
                     f"EMA200: ${row['ema_200']:.2f} | VWAP: ${vwap:.2f}")
            log.info(f"EntryScore: {entry_score}/3 | "
                     f"PSAR={'✓' if entry_details['psar_bullish'] else '✗'} | "
                     f"VolHeat={'✓' if entry_details['extra_high_vol_up'] else '✗'} ({row['vol_heatmap']}) | "
                     f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '✗'}")

            open_trade = trades_col.find_one({"instrument": symbol, "status": "open"})

            # ═════════════════════════════════════════════════════════════════
            # SELL / POSITION MANAGEMENT  (delegated to sell_bot v2.3,
            #   but keep basic stop/trailing here as backup)
            # ═════════════════════════════════════════════════════════════════

            if open_trade:
                entry_price   = open_trade["entry_price"]
                highest_price = open_trade.get("highest_price", entry_price)
                quantity      = open_trade["quantity"]
                time_in_trade = (datetime.now() - open_trade["entry_timestamp"]).total_seconds() / 3600

                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade["_id"], {"highest_price": highest_price})

                profit_pct  = (current_price - entry_price) / entry_price
                pnl_ps      = current_price - entry_price
                current_pnl = pnl_ps * quantity
                update_trade_document(open_trade["_id"], {"current_pnl": current_pnl})

                log.info(f"OPEN | Entry: ${entry_price:.2f} | P&L: {profit_pct:.2%} | Score: {trend_score:.1f}")

                # ── Stop loss ─────────────────────────────────────────────────
                should_exit = False
                exit_reason = None

                if profit_pct <= -STOP_LOSS_1:
                    if trend_score < TREND_SCORE_DROP_EXIT:
                        should_exit = True
                        exit_reason = "stop_loss_10pct_trend_weak"
                        log.info(f"EXIT: 10% loss + score {trend_score:.1f} < {TREND_SCORE_DROP_EXIT}")
                    elif profit_pct <= -STOP_LOSS_2:
                        should_exit = True
                        exit_reason = "stop_loss_15pct_hard"
                        log.info(f"EXIT: 15% hard stop reached")
                    else:
                        log.info(f"Holding at {profit_pct:.2%} loss — trend still strong ({trend_score:.1f})")

                if should_exit and quantity > 0:
                    sell_price = get_bid_price(contract) or current_price
                    order = create_sell_order(quantity, limit_price=sell_price)
                    ib.placeOrder(contract, order)

                    realized = pnl_ps * quantity
                    update_trade_document(open_trade["_id"], {
                        "exit_price":       current_price,
                        "exit_timestamp":   datetime.now(),
                        "reason":           exit_reason,
                        "realized_pnl":     realized,
                        "exit_trend_score": trend_score,
                        "status":           "closed",
                    })
                    log.info(f"STOP LOSS | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                    await asyncio.sleep(0.5)
                    continue

                # ── Take profit ───────────────────────────────────────────────
                if profit_pct >= TAKE_PROFIT_PCT:
                    sell_price = get_bid_price(contract) or current_price
                    order = create_sell_order(quantity, limit_price=sell_price)
                    ib.placeOrder(contract, order)

                    realized = pnl_ps * quantity
                    update_trade_document(open_trade["_id"], {
                        "exit_price":       current_price,
                        "exit_timestamp":   datetime.now(),
                        "reason":           f"take_profit_{profit_pct:.2%}",
                        "realized_pnl":     realized,
                        "exit_trend_score": trend_score,
                        "status":           "closed",
                    })
                    log.info(f"TAKE PROFIT | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                    await asyncio.sleep(0.5)
                    continue

                # ── Trailing stop (only when profitable) ──────────────────────
                if profit_pct > 0:
                    atr_stop  = highest_price - (atr * ATR_MULTIPLIER)
                    pct_stop  = highest_price * (1 - MIN_TRAILING_PCT)
                    fix_stop  = highest_price - BASE_TRAILING_AMT
                    t_factor  = min(1.0, time_in_trade / 24)
                    trail_stop = max(atr_stop * (1 - 0.25 * t_factor), pct_stop, fix_stop)

                    if current_price <= trail_stop and quantity > 0:
                        sell_price = get_bid_price(contract) or current_price
                        order = create_sell_order(quantity, limit_price=sell_price)
                        ib.placeOrder(contract, order)

                        realized = pnl_ps * quantity
                        update_trade_document(open_trade["_id"], {
                            "exit_price":       current_price,
                            "exit_timestamp":   datetime.now(),
                            "reason":           "trailing_stop_hit",
                            "realized_pnl":     realized,
                            "exit_trend_score": trend_score,
                            "status":           "closed",
                        })
                        log.info(f"TRAILING STOP | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                        await asyncio.sleep(0.5)
                        continue

            # ═════════════════════════════════════════════════════════════════
            # BUY / ENTRY LOGIC  (v2.1 — with duplicate prevention)
            # ═════════════════════════════════════════════════════════════════

            else:
                # ── Score check ───────────────────────────────────────────────
                if entry_score < ENTRY_SCORE_MIN:
                    log.info(
                        f"{symbol}: entry score {entry_score}/3 < {ENTRY_SCORE_MIN} — "
                        f"PSAR={'✓' if entry_details['psar_bullish'] else '✗'} | "
                        f"VolHeat={'✓' if entry_details['extra_high_vol_up'] else '✗'} | "
                        f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '✗'} "
                        f"— will re-check next cycle"
                    )
                    await asyncio.sleep(0.5)
                    continue

                log.info(
                    f"{symbol}: ENTRY SCORE {entry_score}/3 ≥ {ENTRY_SCORE_MIN} ✓  "
                    f"PSAR={'✓' if entry_details['psar_bullish'] else '✗'} | "
                    f"VolHeat={'✓' if entry_details['extra_high_vol_up'] else '✗'} | "
                    f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '✗'}"
                )

                # ── Gate 0: MongoDB duplicate check (NEW — authoritative) ────
                if has_open_trade_in_db(symbol):
                    log.info(f"{symbol}: Gate 0 FAILED — open trade exists in DB — skipping")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 0.5: Cooldown check (NEW) ───────────────────────────
                if is_in_cooldown(symbol):
                    log.info(f"{symbol}: COOLDOWN FAILED — sell bot set cooldown — skipping")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 1: IBKR position check (secondary safety net) ───────
                if has_open_position(symbol):
                    log.info(f"{symbol}: Gate 1 FAILED — existing IB position — skipping")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 2: Pending order check ──────────────────────────────
                if has_pending_order(symbol):
                    log.info(f"{symbol}: Gate 2 FAILED — pending IB order — skipping")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 4: Price check ──────────────────────────────────────
                ask_price = get_ask_price(contract)
                if not ask_price:
                    log.info(f"{symbol}: Gate 4 FAILED — no valid ask price — skipping")
                    await asyncio.sleep(0.5)
                    continue

                # ── Calculate position size (v2.1 — dollar-based) ────────────
                order_qty = calculate_order_qty(ask_price)
                log.info(
                    f"{symbol}: position size = {order_qty} shares "
                    f"(${ORDER_BUDGET} budget / ${ask_price:.2f} per share)"
                )

                # ── Place order (RTH-aware) ───────────────────────────────────
                order = create_buy_order(order_qty, limit_price=ask_price)
                ib.placeOrder(contract, order)

                trade_doc = create_trade_document(
                    symbol               = symbol,
                    entry_price          = ask_price,
                    quantity             = order_qty,
                    trend_score_at_entry = trend_score,
                    entry_score          = entry_score,
                    entry_details        = entry_details,
                )
                trades_col.insert_one(trade_doc)

                log.info(
                    f"BUY | {symbol} | ask=${ask_price:.2f} | qty={order_qty} "
                    f"(${order_qty * ask_price:.0f}) | "
                    f"session={'RTH' if is_rth() else 'EXT'} | "
                    f"score={entry_score}/3 | trend={trend_score:.1f} | "
                    f"PSAR ✓ | VolHeat={'✓' if entry_details['extra_high_vol_up'] else '—'} | "
                    f"EMA9>21={'✓' if entry_details['ema_9_21_stack'] else '—'} | "
                    f"G0 ✓ | G1 ✓ | G2 ✓ | G4 ✓"
                )

            await asyncio.sleep(0.5)

        log.info(f"\n{'='*60}")
        log.info("Scan complete. Next scan in 5 minutes.")
        watchlist_print()
        log.info(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# TELEGRAM POLLER
# ══════════════════════════════════════════════════════════════════════════════

def load_session() -> str:
    if os.path.exists(SESSION_FILE):
        with open(SESSION_FILE, encoding="utf-8") as f:
            return f.read().strip()
    return ""


def save_session(s: str):
    with open(SESSION_FILE, "w", encoding="utf-8") as f:
        f.write(s)
    log.info(f"Telegram session saved → {SESSION_FILE}")


async def telegram_poller():
    log.info("Telegram poller started.")

    while True:
        tg = TelegramClient(StringSession(load_session()), API_ID, API_HASH)
        try:
            await tg.start(phone=PHONE)
            save_session(tg.session.save())

            m = re.search(r't\.me/(?:joinchat/|\+)([A-Za-z0-9_-]+)', INVITE_LINK)
            if not m:
                raise ValueError(f"Cannot parse invite hash: {INVITE_LINK}")
            try:
                await tg(ImportChatInviteRequest(m.group(1)))
                log.info("Joined Telegram channel")
            except UserAlreadyParticipantError:
                log.info("Already a member of the channel")

            channel      = await tg.get_entity(INVITE_LINK)
            last_seen_id = None
            log.info(f"Polling channel: {channel.title} (ID: {channel.id})")

            while True:
                messages = await tg.get_messages(channel, limit=1)
                if messages:
                    msg    = messages[0]
                    is_new = (last_seen_id is None or msg.id != last_seen_id)

                    if msg.text and is_new:
                        parsed = parse_structured_message(msg.text)
                        if parsed:
                            symbol = parsed["symbol"]
                            status = parsed["status"]
                        else:
                            symbol = extract_ticker(msg.text)
                            status = "NEW"

                        preview = msg.text[:120] + ("..." if len(msg.text) > 120 else "")
                        safe_print(f"\n[{msg.date}]")
                        safe_print(f"   Message : {preview}")
                        safe_print(f"   Ticker  : {symbol or '(not found)'}")
                        safe_print(f"   Status  : NEW")

                        if symbol and is_new_signal(status):
                            inserted = watchlist_add(symbol, source="telegram")
                            if inserted:
                                safe_print(f"   → Added {symbol} to watchlist_tickers in MongoDB")
                            else:
                                safe_print(f"   → {symbol} already in watchlist_tickers, skipped")
                        elif symbol:
                            log.info(f"{symbol}: status='{status}' — not NEW, skipping watchlist add")

                        last_seen_id = msg.id

                log.debug(f"Telegram: waiting {POLL_INTERVAL}s...")
                await asyncio.sleep(POLL_INTERVAL)

        except FloodWaitError as e:
            log.warning(f"Telegram rate-limited — waiting {e.seconds}s")
            await asyncio.sleep(e.seconds + 5)
        except KeyboardInterrupt:
            log.info("Telegram poller stopped by user.")
            break
        except Exception as e:
            log.error(f"Telegram error: {type(e).__name__}: {e} — reconnecting in 15s")
            await asyncio.sleep(15)
        finally:
            try:
                await tg.disconnect()
            except Exception:
                pass


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

async def main():
    log.info("=== Unified Trading Bot v2.1 Starting ===")
    log.info(f"Position sizing: ${ORDER_BUDGET} per trade "
             f"(min {MIN_ORDER_QTY}, max {MAX_ORDER_QTY} shares)")
    log.info(f"Entry gates: Gate 0 (DB duplicate) → Cooldown → Gate 1 (IBKR pos) "
             f"→ Gate 2 (pending) → Gate 4 (price)")
    ibkr_connect()
    await asyncio.gather(
        telegram_poller(),
        trading_loop(),
    )


if __name__ == "__main__":
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        log.info("Bot stopped by user.")
    finally:
        ib.disconnect()
        log.info("IBKR disconnected.")
